In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from solver import *

# 1D Tests in periodic conditions

## Stationnary equation
### Free of spatial variables

$$\psi\mathrm{Op}(\xi^2 + 1, u)(x) = \sin x$$

Tests the stationary pseudo-differential solver (`solve_stationary_psiOp(order=1)`) on a constant-symbol operator $\xi^2 + 1$ with periodic boundary conditions on $[0, 2\pi)$. For $u(x) = \tfrac{1}{2}\sin x$, the operator yields $\sin x$, verified against the exact solution in both absolute and real components.

In [ ]:
# Define symbolic variables
x, xi = symbols('x xi')
u = Function('u')(x)

# Define the PDE: psiOp[xi^2 + 1, u](x) = 0
equation = Eq(psiOp(xi**2 + 1, u), sin(x))  # rhs will be overwritten

# Define exact solution

def u_exact(x):
    return np.sin(x)/2

# Create the solver
solver = PDESolver(equation)

# Setup parameters
Lx = 2 * np.pi
Nx = 512
solver.setup(Lx=Lx, Nx=Nx, initial_condition=None)

# Solve the stationary problem
u_num = solver.solve_stationary_psiOp(order=1)

# Exact solution on grid
u_ref = u_exact(solver.X)

# Compute relative L2 error
error = np.linalg.norm(np.real(u_num) - u_ref) / np.linalg.norm(u_ref)

solver.test(u_exact=u_exact, threshold=1e-10, component='abs')

solver.show_stationary_solution(u=u_num, component='real')

### Depending on spatial variables

$$\psi\mathrm{Op}(\xi^2 + x^2 + 1, u)(x) = 3(1 - x^2)e^{-x^2}$$

Tests the stationary `psiOp` solver for a spatially varying symbol $p(x,\xi) = \xi^2 + x^2 + 1$ with exact solution $u(x) = e^{-x^2}$ on a wide periodic domain ($L_x = 30$) to suppress boundary wrap-around. After running `solver.test`, the cell computes the first-order asymptotic right inverse $R_1$ and plots the regularizing remainder symbol $K_1(x,\xi) = (P \circ R_1)(x,\xi) - 1$ over $(x, \xi) \in [-5, 5]^2$.

In [ ]:
# Define symbolic variables
x, xi = symbols('x xi', real=True)
u = Function('u')(x)

# Define the symbol
p_symbol = xi**2 + x**2 + 1

# Define the right-hand side f(x) = P[u_exact]
f_expr = 3 * (1 - x**2) * exp(-x**2)

# Define the full equation: P[u] = f
equation = Eq(psiOp(p_symbol, u), f_expr)

# Exact and initial
def u_exact(x): return np.exp(-x**2)


solver = PDESolver(equation)
solver.setup(Lx=30, Nx=1024, initial_condition=None)

u_num = solver.solve_stationary_psiOp(order=1)
solver.test(u_exact=u_exact, threshold=2, component='real')

# 2. Reconstruct the solver and access the internal PseudoDifferentialOperator
# (Assuming solver setup has been run)
total_op = solver.psi_ops[0][1] # Get the PseudoDifferentialOperator instance

# 3. Obtain the Right Inverse Symbol of order M (e.g., order=1)
order = 1
R_symbol = total_op.right_inverse_asymptotic(order=order)

# 4. Compute the composed symbol (P o R) using the symbol composition formula
# For order=1, we need the 0-th and 1st-st derivatives:
comp_0 = p_symbol * R_symbol
comp_1 = -I * diff(p_symbol, xi) * diff(R_symbol, x)
composed_symbol = simplify(comp_0 + comp_1)

# 5. Extract the Regularizing Operator symbol K_M = P o R - 1
regularizer_symbol = simplify(composed_symbol - 1)
print("\n--- Regularizing Operator Symbol K_1(x, xi) ---")
pprint(regularizer_symbol)
K_1_func = lambdify((x, xi), regularizer_symbol, modules='numpy')

# 2. Define the grid
x_vals = np.linspace(-5, 5, 400)
xi_vals = np.linspace(-5, 5, 400)
X, XI = np.meshgrid(x_vals, xi_vals)

# 3. Evaluate the compiled function over the 2D grid
Z = K_1_func(X, XI)

# 4. Plot the error symbol
plt.figure(figsize=(8, 6))
contour = plt.contourf(X, XI, Z, levels=50, cmap='RdBu_r')
plt.colorbar(contour, label='$K_1(x, \\xi)$')
plt.xlabel('$x$ (Space)')
plt.ylabel('$\\xi$ (Frequency)')
plt.title('Error Footprint of the Asymptotic Inverse ($K_1$)')
plt.show()

## Integro-differential equation
### With Op

$$u_t = u_x + \mathrm{Op}\!\left(\frac{1}{i(k_x + \varepsilon)}\right)u - u, \qquad \varepsilon = 10^{-3}$$

Tests `RK4` time integration of an integro-differential equation combining local advection and damping with a Fourier multiplier `Op` term on a periodic domain of length $6\pi$. Starting from $u(x,0) = \sin x$, the exact solution decays exponentially as $u(x,t) = e^{-t}\sin x$.

In [ ]:
# Define the symbols and the integro-differential equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)
eps = 0.001
# Define the equation with an integral term using Op
integral_term = Op(1 / (I * (kx + eps)), u)

eq = Eq(diff(u, t), diff(u, x) + integral_term - u)

# Create the solver
# solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)
solver = PDESolver(eq, time_scheme='RK4')

# Simulation parameters
Lt = 2.0  # Total simulation time
Lx = 6*np.pi   # Spatial domain size
Nx = 1024 # Number of grid points
Nt = 200  # Number of time steps

# Initial condition
initial_condition = lambda x: np.sin(x)

# Setup the solver
solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    initial_condition=initial_condition
)

# Solve the equation
solver.solve()

# Exact solution at any time t
def u_exact(x, t):
    return np.exp(-t) * np.sin(x)

# Automatic testing at multiple times
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=1e-5,  # Adjust threshold if necessary
        component='real'
    )
    for i in range(n_test + 1)
]


### With psiOp

$$u_t = \psi\mathrm{Op}\!\left(i\xi + \frac{1}{i(\xi + \varepsilon)} - 1\right)u, \qquad \varepsilon = 10^{-3}$$

Equivalent to the previous integro-differential equation written inside a single `psiOp` symbol, using the default time integrator with $2/3$ dealiasing on $L_x = 6\pi$ ($N_x = 2048$). For $u(x,0) = \sin x$, the exact solution is $u(x,t) = e^{-t}\sin x$.

In [ ]:
# Define the symbols and the integro-differential equation
t, x, xi, omega = symbols('t x xi omega')
u_func = Function('u')
u = u_func(t, x)
eps = 0.001
# Define the equation with an integro-differential term using psiOp
integral_term = psiOp(I*xi + 1 / (I * (xi + eps)) - 1, u)

eq = Eq(diff(u, t), integral_term)

# Create the solver
# solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Lt = 2.0  # Total simulation time
Lx = 6*np.pi   # Spatial domain size
Nx = 2048 # Number of grid points
Nt = 200  # Number of time steps

# Initial condition
initial_condition = lambda x: np.sin(x)

# Setup the solver
solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    initial_condition=initial_condition
)

# Solve the equation
solver.solve()

# Exact solution at any time t
def u_exact(x, t):
    return np.exp(-t) * np.sin(x)

# Automatic testing at multiple times
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=1e-5,  # Adjust threshold if necessary
        component='real'
    )
    for i in range(n_test + 1)
]


## Convolution equation
### With Op

$$u_t = -\mathrm{Op}\!\big(\widehat{f}_\lambda(k_x)\big)\,u, \qquad f_\lambda(x) = e^{-\lambda |x|}$$

Tests a spatial convolution operator defined via its Fourier transform multiplier on $[0, 2\pi)$ with $\lambda = 1$ and initial condition $u(x,0) = \cos x$. Mode $k$ decays at rate $\frac{2\lambda}{\lambda^2 + k^2}$, yielding the exact solution $u(x,t) = \cos(x)\,e^{-\frac{2\lambda}{\lambda^2 + k^2}t}$ for $k = 1$.

In [ ]:
# Define symbols
t, x, kx, xi, omega, lam = symbols('t x kx xi omega lambda')
u_func = Function('u')
u = u_func(t, x)
lam = 1.0
# Define the kernel f(x) = exp(-lambda * |x|)
f_kernel = exp(-lam * Abs(x))

# Convolution equation: ∂t u = - OpConv(f_kernel, u)
eq = Eq(diff(u, t), -Op(fourier_transform(f_kernel, x, kx/(2*pi)), u))

# Create solver with lambda=1
solver = PDESolver(eq.subs(lam, 1), time_scheme='RK4')

# Simulation parameters
Lt = 2.0
solver.setup(
    Lx=2 * np.pi, Nx=1024, Lt=Lt, Nt=100,
    initial_condition=lambda x: np.cos(x)
)

# Solve
solver.solve()

# Exact solution: u(x, t) = cos(x) * exp(- (2*lambda / (lambda^2 + k^2)) * t )
def u_exact(x, t):
    lam_val = 1.0
    k_val = 1.0  # mode k=1 (cos(x))
    decay = 2 * lam_val / (lam_val**2 + k_val**2)
    return np.cos(x) * np.exp(-decay * t)

# Automatic testing at multiple times
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='abs')
            for i in range(n_test + 1)]


### With psiOp

$$u_t = -\psi\mathrm{Op}\!\big(\widehat{f}_\lambda(\xi)\big)\,u, \qquad f_\lambda(x) = e^{-\lambda |x|}$$

Solves the same convolution equation with $\lambda = 1$ and $u(x,0) = \cos x$ using `psiOp` and the default time integrator on $N_x = 256$ points, verified against $u(x,t) = \cos(x)\,e^{-\frac{2\lambda}{\lambda^2 + k^2}t}$ for $k = 1$.

In [ ]:
# Define symbols
t, x, kx, xi, omega, lam = symbols('t x kx xi omega lambda')
u_func = Function('u')
u = u_func(t, x)
lam = 1.0
# Define the kernel f(x) = exp(-lambda * |x|)
f_kernel = exp(-lam * Abs(x))

# Convolution equation: ∂t u = - OpConv(f_kernel, u)
eq = Eq(diff(u, t), -psiOp(fourier_transform(f_kernel, x, xi/(2*pi)), u))

# Create solver with lambda=1
solver = PDESolver(eq.subs(lam, 1))

# Simulation parameters
Lt = 2.0
solver.setup(
    Lx=2 * np.pi, Nx=256, Lt=Lt, Nt=100,
    initial_condition=lambda x: np.cos(x)
)

# Solve
solver.solve()

# Exact solution: u(x, t) = cos(x) * exp(- (2*lambda / (lambda^2 + k^2)) * t )
def u_exact(x, t):
    lam_val = 1.0
    k_val = 1.0  # mode k=1 (cos(x))
    decay = 2 * lam_val / (lam_val**2 + k_val**2)
    return np.cos(x) * np.exp(-decay * t)

# Automatic testing at multiple times
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='abs')
            for i in range(n_test + 1)]


## Transport equation
### With PDE

$$u_t = -u_x, \qquad u(x,0) = e^{-x^2}$$

Tests pure rightward advection at unit speed using `RK4` on $[-5, 5)$ ($L_x = 10$) with a Gaussian initial pulse. The reference solution wraps periodically across the domain boundaries over $L_t = 5.0$.

In [ ]:
# Define the symbols and the 1D transport equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

eq = Eq(diff(u, t), -diff(u, x))  # Rightward transport at speed +1

# Create the solver
solver = PDESolver(eq, time_scheme='RK4')
Lt=5.0
# Domain setup and initial condition
solver.setup(
    Lx=10, Nx=1024, Lt=Lt, Nt=2000,
    initial_condition=lambda x: np.exp(-x**2)
)

# Solving
solver.solve()

# Exact solution at t = 2.0 (with periodic wrapping)
def u_exact(x, t):
    L = 10  # Domain length
    return np.exp(-((x - t + L/2) % L - L/2)**2)

# Automatic testing
n_test = 5
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

### With Op

$$u_t + c\,u_x = -\mathrm{Op}\!\big(|k_x|^\beta\big)\,u, \qquad c = 1.0,\ \beta = 1.3$$

Tests advection combined with fractional dissipation $|k_x|^\beta$ via `Op` on $[0, 2\pi)$ using `RK4` and dealiasing ratio $0.5$. With $u(x,0) = \cos x$, the exact solution translates and decays as $u(x,t) = \cos(x - ct)\,e^{-t|1|^\beta}$.

In [ ]:
# Define symbols and equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

# Parameters
c = 1.0
beta = 1.3

# Equation
eq = Eq(diff(u, t) + c * diff(u, x), -Op(abs(kx)**beta, u))

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=0.5, time_scheme='RK4')
Lt=2.0
# Domain and initial condition setup
solver.setup(
    Lx=2 * np.pi, Nx=1024, Lt=Lt, Nt=2000,
    initial_condition=lambda x: np.cos(x)
)

# Solve the equation
solver.solve()

# Exact solution function at t = 1
def u_exact(x, t):
    return np.cos(x - c * t) * np.exp(-t * abs(1)**beta)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

### With psiOp

$$u_t = -\psi\mathrm{Op}\!\big(ic\xi + |\xi|^\beta\big)\,u, \qquad c = 1.0,\ \beta = 1.3$$

Combines transport and fractional dissipation into a single `psiOp` symbol on $[0, 2\pi)$ with $2/3$ dealiasing and initial condition $u(x,0) = \cos x$, matching $u(x,t) = \cos(x - ct)\,e^{-t|1|^\beta}$.

In [ ]:
# Define symbols and equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

# Parameters
c = 1.0
beta = 1.3

# Equation
eq = Eq(diff(u, t), -psiOp(c*I*xi + abs(xi)**beta, u))

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=2/3)
Lt=2.0
# Domain and initial condition setup
solver.setup(
    Lx=2 * np.pi, Nx=1024, Lt=Lt, Nt=2000,
    initial_condition=lambda x: np.cos(x)
)

# Solve the equation
solver.solve()

# Exact solution function at t = 1
def u_exact(x, t):
    return np.cos(x - c * t) * np.exp(-t * abs(1)**beta)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

## Heat equation
### with periodic initial condition

$$u_t = u_{xx}, \qquad u(x,0) = \sin x$$

Baseline 1D heat equation on $[0, 2\pi)$ integrated with `RK4` ($N_x = 1024$, $N_t = 100$) and verified against $u(x,t) = e^{-t}\sin x$.

In [ ]:
# Define the symbols and the equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

# 1D heat equation
eq = Eq(diff(u, t), diff(u, x, x)) 

# Create the solver
solver = PDESolver(eq, time_scheme='RK4')
Lt=2.0
# Domain and initial condition setup
solver.setup(
    Lx=2 * np.pi, Nx=1024, Lt=Lt, Nt=100,
    initial_condition=lambda x: np.sin(x)
)

# Solve the equation
solver.solve()

# Exact solution function
def u_exact(x, t):
    return np.sin(x) * np.exp(-t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]


### with compact support initial condition

$$u_t = u_{xx}, \qquad u(x,0) = e^{-2x^2}$$

Tests diffusive spreading of a localized Gaussian ($\sigma = 0.5$) on a wider periodic domain ($L_x = 5\pi$) using `RK4` and renders an interactive JS/HTML animation. The exact solution has linearly growing variance $\sigma^2(t) = \sigma^2 + 2t$:

$$u(x,t) = \frac{1}{\sqrt{1 + 2t/\sigma^2}}\exp\!\left(-\frac{x^2}{2(\sigma^2 + 2t)}\right)$$

In [ ]:
# Definition of symbols and equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u = Function('u')(t, x)

# 1D heat equation
eq = Eq(diff(u, t), diff(u, x, x))

# Creation of the solver with RK4 scheme
solver = PDESolver(eq, time_scheme='RK4')

# Simulation parameters
Lt = 2.0  # Total time
Lx = 5 * np.pi  # Spatial domain
Nx = 512  # Grid points
Nt = 100  # Time steps

# Solver setup with Gaussian initial condition
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    initial_condition=lambda x: np.exp(-2 * x**2)  # Gaussian σ=0.5
)

# Solving the equation
solver.solve()

# Exact solution for the Gaussian with diffusion
def u_exact(x, t):
    sigma = 0.5
    variance = sigma**2 + 2 * t  # Variance increases linearly
    return (np.exp(-x**2 / (2 * variance)) /
            np.sqrt(1 + 2 * t / sigma**2))

# Automated tests at different times
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=1e-3,  
        component='real'
    )
    for i in range(n_test + 1)
]

# Animate the solution
ani = solver.animate()
HTML(ani.to_jshtml())

### With Op

$$u_t = \frac{1}{2}\mathrm{Op}\!\big((ik_x)^2\big)\,u + \frac{1}{2}u_{xx}$$

Splits the 1D Laplacian equally between a Fourier multiplier `Op` term and a standard derivative term on $[0, 2\pi)$ using `RK4` and $u(x,0) = \sin x$, verified against $u(x,t) = e^{-t}\sin x$.

In [ ]:
# Define the symbols and the equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

# 1D heat equation
#eq = Eq(diff(u, t),  diff(u, x, 2)) 
eq = Eq(diff(u, t), Op((I*kx)**2, u)/2 + diff(u, x, 2)/2)
#eq = Eq(Op(-(I*omega),u), Op(-(I*kx)**2, u)/2 + diff(u, x, 2)/2)
# Create the solver
solver = PDESolver(eq, time_scheme='RK4')
Lt=2.0
# Domain and initial condition setup
solver.setup(
    Lx=2 * np.pi, Nx=512, Lt=2.0, Nt=100,
    initial_condition=lambda x: np.sin(x)
)

# Solve the equation
solver.solve()

# Exact solution function at t = 1
def u_exact(x, t):
    return np.sin(x) * np.exp(-t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-3, component='real') for i in range(n_test + 1)]


### In full Op formulation

$$\mathrm{Op}(-i\omega)\,u = \mathrm{Op}\!\big((-ik_x)^2\big)\,u$$

Tests the heat equation expressed in frequency-space `Op` form on $[0, 2\pi)$ using `RK4` ($N_x = 512$, $N_t = 400$) and $u(x,0) = \sin x$, verified against $u(x,t) = e^{-t}\sin x$.

In [ ]:
# Define the symbols and the equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

# 1D heat equation in Op formulation
# eq = Eq(Op((-I*omega), u), diff(u, x, x))
eq = Eq(Op((-I*omega), u), Op((-I*kx)**2, u))

# Create the solver
solver = PDESolver(eq, time_scheme='RK4')
Lt=2.0
# Domain and initial condition setup
solver.setup(
    Lx=2 * np.pi, Nx=512, Lt=Lt, Nt=400,
    initial_condition=lambda x: np.sin(x),
    initial_velocity=lambda x: np.zeros_like(x)  # ∂u/∂t(x,0) = 0
)

# Solve the equation
solver.solve()

# Exact solution function
def u_exact(x, t):
    return np.sin(x) * np.exp(-t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

### With fractional Op

$$u_t = -\mathrm{Op}\!\big(|k_x|^\alpha\big)\,u + \nu\,u_{xx}, \qquad \alpha = 1.8,\ \nu = 0.1$$

Combines fractional diffusion $|k_x|^\alpha$ and standard viscosity $\nu u_{xx}$ on $[0, 2\pi)$ using `RK4` and dealiasing ratio $0.5$. For $u(x,0) = \sin x$, the exact solution decays at rate $|1|^\alpha + \nu\cdot 1^2$: $u(x,t) = \sin(x)\,e^{-t(|1|^\alpha + \nu)}$.

In [ ]:
# Define symbols and equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

# Parameters
alpha = 1.8
nu = 0.1

# Equation
eq = Eq(diff(u, t), -Op(abs(kx)**alpha, u) + nu * diff(u, x, 2))

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=0.5, time_scheme='RK4')
Lt=2.0
# Domain and initial condition setup
solver.setup(
    Lx=2 * np.pi, Nx=256, Lt=Lt, Nt=2000,
    initial_condition=lambda x: np.sin(x)
)

# Solve the equation
solver.solve()

# Exact solution function at t = 1
def u_exact(x, t):
    return np.sin(x) * np.exp(-t * (abs(1)**alpha + nu * 1**2))

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

### With source term

$$u_t = u_{xx} + 3e^{-t}\sin(2x) + (t^2 + 2t)\cos x$$

Tests inhomogeneous time- and space-dependent forcing on $[0, 4\pi)$ using `RK4` ($N_x = 1024$, $N_t = 2000$) with $u(x,0) = \sin(2x)$, compared against the exact solution $u(x,t) = e^{-t}\sin(2x) + t^2\cos x$.

In [ ]:
# Define the symbols and the heat equation with a source term
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

# Source term
source_term = 3 * exp(-t) * sin(2*x) + (t**2 + 2*t) * cos(x)

# Equation with source term
eq = Eq(diff(u, t), diff(u, x, x) + source_term)

# Create the solver with RK4 scheme
solver = PDESolver(eq, time_scheme="RK4")

# Simulation parameters
Lt = 5.0  # Total simulation time
solver.setup(
    Lx=4 * np.pi, Nx=1024, Lt=Lt, Nt=2000,
    initial_condition=lambda x: np.sin(2 * x),
    initial_velocity=lambda x: np.zeros_like(x)
)

# Solving
solver.solve()


# Exact solution
def u_exact(x, t):
    return np.exp(-t) * np.sin(2*x) + t**2 * np.cos(x)

# Automatic testing
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=3,  # Adjust threshold if necessary
        component='real'
    )
    for i in range(n_test + 1)
]

### Hermite functions: heat equation with a harmonic potential

$$u_t = u_{xx} - x^2\,u$$

The Schrödinger harmonic oscillator in imaginary time. Its eigenfunctions are the Hermite functions $H_n(x)e^{-x^2/2}$ with eigenvalues $-(2n+1)$, so any superposition has an exact solution:

$$u(x,t) = \sum_n c_n\,e^{-(2n+1)t}\,H_n(x)\,e^{-x^2/2}$$

Here $u(x,0) = H_0 e^{-x^2/2} + 0.7\,H_3(x)e^{-x^2/2}$: the two components decay at very different rates ($e^{-t}$ and $e^{-7t}$). The potential is a zero-order variable coefficient, treated explicitly: the error decreases like $\Delta t^4$ (2.8e-10, 1.8e-11, 2.3e-12 for $N_t$ = 500, 1000, 2000).
**Requires the variable-coefficient fix in `_parse_equation`** (zero-order terms such as $x^2u$ *and* derivative terms such as $x\,u_x$, $a(x)u_{xx}$). Without it, a zero-order variable coefficient is silently replaced by $1$ and a variable coefficient in front of a derivative crashes with a `TypeError`. With the fix these terms are evaluated pointwise, like nonlinear terms (explicit part of the integrator).

In [ ]:
# Define the symbols and the equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

from numpy.polynomial.hermite import hermval

eq = Eq(diff(u, t) - diff(u, x, x) + x**2 * u, 0)   # u_t = u_xx - x^2 u

# Exact solution
def u_exact(x, t):
    return sum(c * np.exp(-(2 * n + 1) * t) * hermval(x, [0] * n + [1]) * np.exp(-x**2 / 2)
               for c, n in ((1.0, 0), (0.7, 3))) # (coefficient c_n, order n) Superposition of the Hermite functions n = 0 and n = 3

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 16
Nx = 128
Lt = 1.0
Nt = 2000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

## Diffusion equation
### With fractional Op

$$u_t = -\mathrm{Op}\!\big(|k_x|^\alpha\big)\,u, \qquad \alpha = 1.5$$

Tests pure fractional diffusion on $[0, 2\pi)$ using `RK4`, dealiasing ratio $0.5$, and $N_t = 2000$ steps starting from $u(x,0) = \sin x$, verified against $u(x,t) = \sin(x)\,e^{-t|1|^\alpha}$.

In [ ]:
# Define the symbols and the equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

# Fractional diffusion equation with alpha = 1.5 and added dissipation
alpha = 1.5
nu = 0.01  # Diffusion coefficient
# eq = Eq(diff(u, t), Op(abs(kx)**alpha, u) + nu * diff(u, x, 2))
eq = Eq(diff(u, t), -Op(abs(kx)**alpha, u))

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=0.5, time_scheme='RK4')
Lt=2.0
# Domain and initial condition setup
solver.setup(
    Lx=2 * np.pi, Nx=256, Lt=Lt, Nt=2000,  # Reduced dt by increasing Nt
    initial_condition=lambda x: np.sin(x)
)

# Solve the equation
solver.solve()

# Exact solution function at t = 1
def u_exact(x, t):
    return np.sin(x) * np.exp(-t * abs(1)**alpha)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

### With psiOp

$$u_t = \psi\mathrm{Op}(-\xi^2, u)$$

Tests the standard diffusion operator $\partial_x^2$ written via `psiOp` with symbol $-\xi^2$ on $[0, 2\pi)$ ($N_x = 2048$, $N_t = 200$) starting from $u(x,0) = \sin x$, compared against $u(x,t) = e^{-t}\sin x$.

In [ ]:
# Definition of the PDE: ∂u/∂t = - ∂²u/∂x² (diffusion equation)
u = Function('u')
t, x, xi = symbols('t x xi')
eq = Eq(diff(u(t,x), t), psiOp(-xi**2, u(t,x)))

# Solver configuration
solver = PDESolver(eq)
Lt=2.0
solver.setup(Lx=2*np.pi, Nx=2048, Lt=Lt, Nt=200,
             initial_condition=lambda x: np.sin(x))

# Solving and visualization
solver.solve()

# Exact solution function
def u_exact(x, t):
    return np.sin(x) * np.exp(-t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]


### Fractional diffusion (`psiOp`, non-smooth symbol): exact solutions on the circle

$$u_t = -(-\Delta)^{\alpha/2}u\quad\Longleftrightarrow\quad u_t = \mathrm{Op}\big(-|\xi|^{\alpha}\big)\,u$$

The problem is linear, so the exponential integrator is exact in time and the solution is spectral.

1. **$\alpha = 1$: Poisson kernel.** With $u(x,0) = \dfrac{1 - r_0^2}{1 - 2r_0\cos x + r_0^2}$ and $r_0 = e^{-a_0}$, the half-Laplacian semigroup is the harmonic extension: $u(x,t) = \dfrac{1 - r^2}{1 - 2r\cos x + r^2}$ with $r = e^{-(a_0 + t)}$.
2. **$\alpha = 3/2$: Fourier series.** With $u(x,0) = e^{\cos x - 1} = \sum_k \tilde I_{|k|}(1)\,e^{ikx}$: $u(x,t) = \sum_k \tilde I_{|k|}(1)\,e^{-|k|^{3/2}t}\,e^{ikx}$.

#### alpha = 1 (Poisson kernel)

In [ ]:
# Define the symbols and the half-Laplacian equation (psiOp form)
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

linear_symbol = -Abs(xi)                                  # -(-Laplacian)^(1/2)
eq = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)))   # u_t = -|D| u

# Exact solution (Poisson kernel on the circle)
a0 = 0.5
def u_exact(x, t):
    r = np.exp(-(a0 + t))
    return (1 - r**2) / (1 - 2 * r * np.cos(x) + r**2)

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi
Nx = 128
Lt = 2.0
Nt = 200

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-11, component='real') for i in range(n_test + 1)]

#### alpha = 3/2 (Fourier series)

In [ ]:
# Fractional exponent
from scipy.special import ive 
alpha = 3 / 2
linear_symbol = -Abs(xi)**Rational(3, 2)
eq = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)))   # u_t = -(-Laplacian)^(3/4) u

# Exact solution: sum_k ive(|k|, 1) exp(-|k|^alpha t) exp(i k x)   (u0 = exp(cos x - 1))
def u_exact(x, t):
    x = np.asarray(x)[None, :]
    K = 40
    k = np.arange(-K, K + 1)[:, None]
    c_k = ive(np.abs(k), 1.0)
    return np.real(np.sum(c_k * np.exp(-np.abs(k)**alpha * t) * np.exp(1j * k * x), axis=0))

# Initial condition
initial_condition = lambda x: np.exp(np.cos(x) - 1)

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Nx = 64
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)
solver.solve()

n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-11, component='real') for i in range(n_test + 1)]

## Burgers' equation

### Periodic Cole–Hopf solution
$$u_t + u\,u_x = \nu\,u_{xx}$$

Exact smooth periodic solution (Cole–Hopf applied to $\varphi = 1 + a\,e^{-\nu k^2 t}\cos kx$, $|a|<1$):

$$u(x,t) = \frac{2\nu a k\,e^{-\nu k^2 t}\sin kx}{1 + a\,e^{-\nu k^2 t}\cos kx}$$

It is exactly periodic on $L_x = 2\pi/k$, so there is no wrap-around error. It tests a quadratic nonlinearity with a first derivative, with the RK4 scheme and with the first-order default scheme.

#### RK4

In [ ]:
# Define the symbols and the Burgers equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

nu = 0.1
eq = Eq(diff(u, t) + u * diff(u, x) - nu * diff(u, x, x), 0)   # u_t + u u_x = nu u_xx

# Exact solution (periodic Cole-Hopf)
a, k = 0.8, 1.0   # |a| < 1 keeps the denominator positive
def u_exact(x, t):
    E = np.exp(-nu * k**2 * t)
    return 2 * nu * a * k * E * np.sin(k * x) / (1 + a * E * np.cos(k * x))

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (quadratic nonlinearity: 2/3 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi / k
Nx = 128
Lt = 3.0
Nt = 300

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-9, component='real') for i in range(n_test + 1)]

#### Default scheme (first order, exponential Euler)

Same equation and exact solution. The error decreases like $1/N_t$, hence the looser threshold.

In [ ]:
# Define the symbols and the Burgers equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

nu = 0.1
eq = Eq(diff(u, t) + u * diff(u, x) - nu * diff(u, x, x), 0)   # u_t + u u_x = nu u_xx

# Exact solution (periodic Cole-Hopf)
a, k = 0.8, 1.0   # |a| < 1 keeps the denominator positive
def u_exact(x, t):
    E = np.exp(-nu * k**2 * t)
    return 2 * nu * a * k * E * np.sin(k * x) / (1 + a * E * np.cos(k * x))

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (quadratic nonlinearity: 2/3 rule)
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi / k
Nx = 128
Lt = 3.0
Nt = 300

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-3, component='real') for i in range(n_test + 1)]

### Two-mode Cole–Hopf solution (steep fronts)

$$u_t + u\,u_x = \nu\,u_{xx}$$

With $\varphi = 1 + a_1 e^{-\nu t}\cos x + a_2 e^{-4\nu t}\cos 2x$ (a solution of the heat equation, positive for the values below), $u = -2\nu\,\varphi_x/\varphi$ is an exact periodic solution:

$$u = \frac{2\nu\left(a_1 e^{-\nu t}\sin x + 2a_2 e^{-4\nu t}\sin 2x\right)}{\varphi}$$

With $a_1 = 0.7$, $a_2 = 0.2$ and $\nu = 0.05$, $\varphi$ gets close to zero: the profile steepens into thin fronts before viscosity smooths it, a much harder spectral test than the single-mode case.

In [ ]:
# Define the symbols and the Burgers equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

nu = 0.05
eq = Eq(diff(u, t) + u * diff(u, x) - nu * diff(u, x, x), 0)   # u_t + u u_x = nu u_xx

# Exact solution (Cole-Hopf, two modes)
a1, a2 = 0.7, 0.2   # phi stays positive: 1 - a2 - a1^2/(8 a2) > 0
def u_exact(x, t):
    E1, E2 = np.exp(-nu * t), np.exp(-4 * nu * t)
    phi = 1 + a1 * E1 * np.cos(x) + a2 * E2 * np.cos(2 * x)
    phi_x = -a1 * E1 * np.sin(x) - 2 * a2 * E2 * np.sin(2 * x)
    return -2 * nu * phi_x / phi

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi
Nx = 256
Lt = 5.0
Nt = 1000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

### Burgers equation with a small viscosity: sinusoidal initial data (Cole, 1951)

$$u_t + u\,u_x = \nu\,u_{xx},\qquad u(x,0) = a\sin x$$

The solver requires a non-null linear operator, so the inviscid equation is not allowed: a small Laplacian regularization $\nu\,u_{xx}$ is mandatory. Here it is the physical viscosity, and the problem keeps an **exact solution**. With the Cole–Hopf transformation $u = -2\nu\,\partial_x\ln\varphi$ and the Jacobi–Anger expansion $e^{z\cos x} = I_0(z) + 2\sum_{n\ge1} I_n(z)\cos nx$, $z = a/2\nu$:

$$u(x,t) = \frac{4\nu\sum_{n\ge1} n\,I_n(z)\,e^{-\nu n^2 t}\sin nx}{I_0(z) + 2\sum_{n\ge1} I_n(z)\,e^{-\nu n^2 t}\cos nx}$$

It goes through the whole life of a Burgers solution: steepening (the inviscid breaking time would be $t^\star = 1/a = 2$), a smooth viscous shock, then decay.

**Conditioning of the exact solution:** $\varphi_{\min} = e^{-a/\nu}$, so the Bessel series loses digits as $\nu$ decreases (about $10^{-12}$ accurate for $\nu = 0.04$, only $10^{-6}$ for $\nu = 0.02$ in double precision). That is why $\nu = 0.04$ is used here.

In [ ]:
# Define the symbols and the viscous Burgers equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)
from scipy.special import ive 

nu = 0.04   # Laplacian regularization (mandatory: the linear operator cannot be null)
eq = Eq(diff(u, t) + u * diff(u, x) - nu * diff(u, x, x), 0)   # u_t + u u_x = nu u_xx

# Exact solution (Cole 1951): Cole-Hopf + Bessel series, u(x, 0) = a sin(x)
def u_exact(x, t):
    a = 0.5
    z = a / (2 * nu)
    n = np.arange(1, 301)[:, None]
    I_n = ive(n, z)     # Exponentially scaled Bessel functions: the common factor e^z cancels
    I_0 = ive(0, z)
    x = np.asarray(x)[None, :]
    E = np.exp(-nu * n**2 * t)
    phi = I_0 + 2 * np.sum(I_n * E * np.cos(n * x), axis=0)
    num = 4 * nu * np.sum(n * I_n * E * np.sin(n * x), axis=0)
    return num / phi

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi
Nx = 256           # Nx = 128 does not resolve the viscous shock after t = 2
Lt = 4.0           # Beyond the inviscid breaking time t* = 1/a = 2
Nt = 2000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

### Inviscid Burgers before shock formation (method of characteristics)

$$u_t + u\,u_x = 0,\qquad u(x,0) = a\sin x$$

There is no dissipation at all (the linear operator is zero). The solution is constant along characteristics, hence it is given implicitly by

$$u = a\sin(x - u\,t)$$

which is solved to machine precision by Newton iterations. It stays smooth until the breaking time $t^\star = 1/a$ (here $t^\star = 2$); the test stops at $L_t = 1.5$. Without viscosity the gradients keep steepening, so the resolution needed grows quickly as $t\to t^\star$ ($N_x = 512$ here; with $N_x = 128$ the error reaches 2e-5 at $t = 1.5$).

**Note:** a zero linear operator needs a one-line fix in `_step_ETD_RK4_order1`: `L_fft = np.asarray(L_fft) + np.zeros(np.shape(self.KX))` right after `L_fft` is computed.

In [ ]:
# Define the symbols and the inviscid Burgers equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(diff(u, t) + u * diff(u, x) + 0.0001*diff(u, x, x) , 0)   # u_t + u u_x = 0

# Exact solution: u = a sin(x - u t), solved by Newton iterations (valid for t < 1/a)
a = 0.5
def u_exact(x, t):
    v = a * np.sin(x)
    for _ in range(100):
        F = v - a * np.sin(x - v * t)
        dF = 1 + a * t * np.cos(x - v * t)
        dv = F / dF
        v = v - dv
        if np.max(np.abs(dv)) < 1e-15:
            break
    return v

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi
Nx = 512
Lt = 1.5          # 75 % of the breaking time t* = 2
Nt = 3000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-3, component='real') for i in range(n_test + 1)]

## Equation depending only on spatial variables

$$u_t = -\psi\mathrm{Op}(1 + \sin x, u) = -(1 + \sin x)\,u$$

Tests a purely spatial symbol $1 + \sin x$ inside `psiOp` with periodic boundary conditions on $[0, 2\pi)$ and initial condition $u(x,0) = \cos x$. Because the operator acts pointwise as multiplication by $-(1 + \sin x)$, the exact solution is $u(x,t) = \cos(x)\,e^{-t(1 + \sin x)}$.

In [ ]:
# === Symbols ===
t, x, xi = symbols('t x xi', real=True)
u = Function('u')
# === Symbol dependent on x, periodic
symbol_expr = 1 + sin(x)
# === Equation ∂t u = -ψOp(symbol_expr, u) = - (1 + sin x) * u
eq = Eq(diff(u(t,x), t), -psiOp(symbol_expr, u(t,x)))
# === Solver creation
solver = PDESolver(eq)
# === Parameters
Lx = 2 * np.pi
Nx = 512
Lt = 2.0
Nt = 200
# === Initial condition
initial_condition = lambda x: np.cos(x)
# === Exact solution
def u_exact(x, t):
    return np.cos(x) * np.exp(-t * (1 + np.sin(x)))
# === Setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition
)
# === Solving
solver.solve()
# === Tests
n_test = 4
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    solver.test(
        u_exact=u_exact,
        t_eval=t_eval,
        threshold=5e-9,
        component='real'
    )


## Wave equation
### With periodic initial condition

$$u_{tt} = u_{xx}, \qquad u(x,0) = \sin x,\quad u_t(x,0) = 0$$

Tests second-order-in-time wave propagation of a single spatial mode on $[0, 2\pi)$ using `RK4`, plotting energy conservation and comparing against $u(x,t) = \sin x \cos t$.

In [ ]:
# Define the symbols and the equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

# Wave equation
eq = Eq(diff(u, t, t), diff(u, x, x))

# Create the solver
solver = PDESolver(eq, time_scheme="RK4")
Lt=2.0
# Domain and initial conditions setup
solver.setup(
    Lx=2 * np.pi, Nx=1024, Lt=Lt, Nt=500,
    initial_condition=lambda x: np.sin(x),
    initial_velocity=lambda x: np.zeros_like(x)  # ∂u/∂t(x,0) = 0
)

# Solving
solver.solve()

solver.plot_energy() # (log=True) 

# Exact solution 
def u_exact(x, t):
    return np.sin(x) * np.cos(t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=5e-2, component='real') for i in range(n_test + 1)]


### With compact support initial condition

$$u_{tt} = u_{xx}, \qquad u(x,0) = e^{-(x - x_0)^2/(2\sigma^2)},\quad u_t(x,0) = 0$$

Tests d'Alembert splitting of a Gaussian pulse ($\sigma = 2.0$, $x_0 = 0$) into left- and right-moving waves on a wide periodic domain ($L_x = 30$) up to $L_t = 10.0$ using `RK4`:

$$u(x,t) = \frac{1}{2}\left[\exp\!\left(-\frac{(x - t - x_0)^2}{2\sigma^2}\right) + \exp\!\left(-\frac{(x + t - x_0)^2}{2\sigma^2}\right)\right]$$

In [ ]:
# Define the symbols and the wave equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u = Function('u')(t, x)

# 1D wave equation: ∂²u/∂t² = ∂²u/∂x²
eq = Eq(diff(u, t, t), diff(u, x, x))

# Create the PDE solver with RK4 time integration
solver = PDESolver(eq, time_scheme="RK4")

# Simulation parameters
Lt = 10.0          # Total simulation time
Lx = 30.0          # Spatial domain size (increased for better Gaussian containment)
Nx = 1024          # Number of spatial grid points
Nt = 1000          # Number of time steps

# Gaussian initial condition parameters
sigma = 2.0        # Standard deviation of the Gaussian
x0 = 0.0           # Center position

# Initial condition: Gaussian pulse
initial_condition = lambda x: np.exp(-(x - x0)**2 / (2*sigma**2))

# Exact solution using d'Alembert's formula for wave equation
def u_exact(x, t):
    return 0.5 * (np.exp(-(x - t - x0)**2 / (2*sigma**2)) + 
                  np.exp(-(x + t - x0)**2 / (2*sigma**2)))

# Setup the solver with increased domain size
solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    initial_condition=initial_condition,
    initial_velocity=lambda x: np.zeros_like(x)  # Zero initial velocity
)

# Solve the equation
solver.solve()

# Visualize energy conservation
solver.plot_energy() # (log=True)

# Perform automatic validation tests
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=2e-2,    # Adjusted threshold for Gaussian decay
        component='real'
    )
    for i in range(n_test + 1)
]


### With a source term

$$u_{tt} = u_{xx} - u + \cos x\big(3\cos(\sqrt{2}\,t) - \sqrt{2}\sin(\sqrt{2}\,t)\big)$$

Tests a forced Klein–Gordon/wave-type equation with $u(x,0) = \cos x$ and $u_t(x,0) = 0$ on $L_x = 6\pi$ using `RK4`, tracking energy and comparing against $u(x,t) = \cos x\left[\left(\frac{t}{2}+1\right)\cos(\sqrt{2}\,t) + \left(\frac{3\sqrt{2}\,t}{4} - \frac{\sqrt{2}}{4}\right)\sin(\sqrt{2}\,t)\right]$.

In [ ]:
# Define the symbols and the wave equation with a source term
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

# Source term
source_term = cos(x) * (3 * cos(sqrt(2) * t) - sqrt(2) * sin(sqrt(2) * t))

# Equation with source term
eq = Eq(diff(u, t, t), diff(u, x, x) - u + source_term)
# eq = Eq(diff(u, t, t), psiOp((I*xi)**2 - 1, u) + source_term)

# Create the solver with RK4 scheme
solver = PDESolver(eq, time_scheme="RK4")
# solver = PDESolver(eq)

# Simulation parameters
Lt = 2.0  # Total simulation time
solver.setup(
    Lx=6 * np.pi, Nx=2048, Lt=Lt, Nt=400,
    initial_condition=lambda x: np.cos(x),
    initial_velocity=lambda x: np.zeros_like(x)
)

# Solving
solver.solve()

# Plot energy evolution
solver.plot_energy() # (log=True)

# Exact solution
def u_exact(x, t):
    return np.cos(x) * ((t/2 + 1) * np.cos(np.sqrt(2) * t) + (3 * np.sqrt(2) * t/4 - np.sqrt(2)/4)*np.sin(np.sqrt(2)*t))

# Automatic testing
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=5e-2,  # Adjust threshold if necessary
        component='real'
    )
    for i in range(n_test + 1)
]

### With psiOp and source term

$$u_{tt} = \psi\mathrm{Op}\big((i\xi)^2 - 1, u\big) + \cos x\big(3\cos(\sqrt{2}\,t) - \sqrt{2}\sin(\sqrt{2}\,t)\big)$$

Solves the same forced equation as above using `psiOp` for the linear spatial operator $(i\xi)^2 - 1$ and the default integrator on $L_x = 6\pi$ ($N_x = 1024$, $N_t = 400$).

In [ ]:
# Define the symbols and the wave equation with a source term
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

# Source term
source_term = cos(x) * (3 * cos(sqrt(2) * t) - sqrt(2) * sin(sqrt(2) * t))

# Equation with source term
# eq = Eq(diff(u, t, t), diff(u, x, x) - u + source_term)
eq = Eq(diff(u, t, t), psiOp((I*xi)**2 - 1, u) + source_term)

# Create the solver with RK4 scheme
# solver = PDESolver(eq, time_scheme="RK4")
solver = PDESolver(eq)

# Simulation parameters
Lt = 2.0  # Total simulation time
solver.setup(
    Lx=6 * np.pi, Nx=1024, Lt=Lt, Nt=400,
    initial_condition=lambda x: np.cos(x),
    initial_velocity=lambda x: np.zeros_like(x)
)

# Solving
solver.solve()

# Plot energy evolution
solver.plot_energy() # (log=True)

# Exact solution
def u_exact(x, t):
    return np.cos(x) * ((t/2 + 1) * np.cos(np.sqrt(2) * t) + (3 * np.sqrt(2) * t/4 - np.sqrt(2)/4)*np.sin(np.sqrt(2)*t))

# Automatic testing
n_test = 4
test_set = [
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=5e-5,  # Adjust threshold if necessary
        component='real'
    )
    for i in range(n_test + 1)
]

### With Op

$$u_{tt} - u_{xx} = -\mathrm{Op}\!\big(|k_x|^\gamma\big)\,u, \qquad \gamma = 1.4$$

Tests a wave equation with fractional spatial operator $|k_x|^\gamma$ via `Op` on $[0, 2\pi)$ using `RK4`, dealiasing ratio $0.5$, $u(x,0) = \sin x$, and $u_t(x,0) = 0$. For mode $k = 1$, the exact solution oscillates at frequency $\sqrt{1 + |1|^\gamma}$: $u(x,t) = \sin(x)\cos\!\left(\sqrt{1 + |1|^\gamma}\,t\right)$.

In [ ]:
# Define symbols and equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

# Parameters
gamma = 1.4

# Equation
eq = Eq(diff(u, t, 2) - diff(u, x, 2), -Op(abs(kx)**gamma, u))

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=0.5, time_scheme='RK4')
Lt=2.0
# Domain and initial condition setup
solver.setup(
    Lx=2 * np.pi, Nx=1024, Lt=Lt, Nt=2000,
    initial_condition=lambda x: np.sin(x),
    initial_velocity=lambda x: np.zeros_like(x)  # Correction ici
)

# Solve the equation
solver.solve()

solver.plot_energy() #(log=True) 

# Exact solution function at t = 1
def u_exact(x, t):
    return np.sin(x) * np.cos(np.sqrt(1 + abs(1)**gamma) * t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=5e-3, component='real') for i in range(n_test + 1)]

### With psiOp

$$u_{tt} = \psi\mathrm{Op}\big((-i\xi)^2 - |\xi|^\gamma, u\big), \qquad \gamma = 1.4$$

Solves the same fractional wave equation using `psiOp` and the default integrator on $[0, 2\pi)$ over $L_t = 5.0$ ($N_x = 512$, $N_t = 1000$), verified at $11$ checkpoints against $u(x,t) = \sin(x)\cos\!\left(\sqrt{1 + |1|^\gamma}\,t\right)$.

In [ ]:
# Define the symbols
t, x, xi = symbols('t x xi', real=True)
u_func = Function('u') 
u = u_func(t, x)

# Wave equation with psiOp
eq = Eq(diff(u, t, t), psiOp((-I*xi)**2 - Abs(xi)**gamma, u))

# Create the solver
solver = PDESolver(eq)

Lt = 5.0

# Domain and initial conditions setup
solver.setup(
    Lx=2 * np.pi, Nx=512, Lt=Lt, Nt=1000,
    initial_condition=lambda x: np.sin(x),
    initial_velocity=lambda x: np.zeros_like(x)  # ∂u/∂t(x,0) = 0
)

# Solve
solver.solve()

# Plot energy
solver.plot_energy() #(log=True)

# Exact solution
def u_exact(x, t):
    return np.sin(x) * np.cos(np.sqrt(1 + abs(1)**gamma) * t)

# Automatic testing
n_test = 10
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=5e-3, component='real') for i in range(n_test + 1)]


## Damped wave equation
### Damped wave with many modes (over- and under-damped)

$$u_{tt} + \gamma\,u_t = u_{xx}, \qquad \gamma = 3.0,\quad u(x,0) = e^{\cos x - 1},\quad u_t(x,0) = 0$$

Tests a multi-mode damped wave on $[0, 2\pi)$ using `RK4` ($N_x = 64$, $N_t = 1000$) where low wavenumbers ($k^2 < \gamma^2/4$) are over-damped while higher wavenumbers ($k^2 > \gamma^2/4$) are under-damped. The exact solution sums the Fourier modes weighted by exponentially scaled modified Bessel functions $\tilde{I}_{|k|}(1)$.

In [ ]:
# ==============================================================================
# Example 1: Damped wave with many modes (over- and under-damped)
# Equation: u_tt + 3*u_t = u_xx
# ==============================================================================
print("\n--- Example 1: Damped wave (over- and under-damped modes) ---")
t, x = symbols('t x')
u_func = Function('u')
u = u_func(t, x)
from scipy.special import ive 

eq = Eq(diff(u, t, 2) + 3 * diff(u, t) - diff(u, x, 2), 0)
solver = PDESolver(eq, time_scheme="RK4")
Lx, Nx = 2 * np.pi, 64
Lt, Nt = 2.0, 1000


# ==============================================================================
# Shared Helper Function for Exact Solutions (Damped Wave Modes)
# ==============================================================================
def mode(k, c, gam, T, kind='cos'):
    """
    Exact solution for u'' + gam*u' + c^2*k^2*u = 0.
    kind='cos': u(0)=1, u'(0)=0
    kind='sin': u(0)=0, u'(0)=1
    """
    wd = np.sqrt(complex(c**2 * k**2 - gam**2 / 4))
    sinc = np.where(abs(wd * T) < 1e-9, T, np.sin(wd * T) / np.where(wd == 0, 1, wd))
    if kind == 'cos':
        return np.real(np.exp(-gam * T / 2) * (np.cos(wd * T) + gam / 2 * sinc))
    return np.real(np.exp(-gam * T / 2) * sinc)
    
def u_exact(X, T):
    K = 40
    ks = np.arange(-K, K + 1)[:, None]
    ck = ive(np.abs(ks), 1.0)
    modes = np.array([mode(abs(k), 1.0, 3.0, T) for k in ks[:, 0]])[:, None]
    return np.sum(ck * modes * np.exp(1j * ks * np.asarray(X)[None, :]), axis=0).real

solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    initial_condition=lambda X: np.exp(np.cos(X) - 1),
    initial_velocity=lambda X: np.zeros_like(X),
    plot=True
)
solver.solve()
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='abs')
            for i in range(n_test + 1)]

### Critically damped single mode

$$u_{tt} + 2\,u_t = u_{xx}, \qquad u(x,0) = \cos x,\quad u_t(x,0) = 0$$

Tests critical damping ($ck = \gamma/2 = 1$) for mode $k = 1$ on $[0, 2\pi)$ using `RK4`, where the exact solution exhibits polynomial-exponential decay $u(x,t) = e^{-t}(1 + t)\cos x$.

In [ ]:
# ==============================================================================
# Example 2: Critically damped single mode
# Equation: u_tt + 2*u_t = u_xx  (where c*k = gamma/2)
# ==============================================================================
print("\n--- Example 2: Critically damped single mode ---")
eq = Eq(diff(u, t, 2) + 2 * diff(u, t) - diff(u, x, 2), 0)
solver = PDESolver(eq, time_scheme="RK4")

Lx, Nx = 2 * np.pi, 64
Lt, Nt = 2.0, 1000

def u_exact(X, T):
    return np.exp(-T) * (1 + T) * np.cos(X)

solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    initial_condition=lambda X: np.cos(X),
    initial_velocity=lambda X: np.zeros_like(X),
    plot=True
)

solver.solve()

n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='abs')
            for i in range(n_test + 1)]

### Initial velocity, underdamped

$$u_{tt} + 0.8\,u_t = u_{xx}, \qquad u(x,0) = 0,\quad u_t(x,0) = \cos x$$

Tests non-zero initial velocity in the under-damped regime ($\gamma = 0.8 < 2ck$) on $[0, 2\pi)$ over $L_t = 10.0$ using `RK4`. Because $u(x,0) \equiv 0$, `norm='absolute'` is used to avoid division by zero at $t = 0$.

In [ ]:
# ==============================================================================
# Example 3: Initial velocity, underdamped
# Equation: u_tt + 0.8*u_t = u_xx
# ==============================================================================
print("\n--- Example 3: Initial velocity (underdamped) ---")
eq = Eq(diff(u, t, 2) + 0.8 * diff(u, t) - diff(u, x, 2), 0)
solver = PDESolver(eq, time_scheme="RK4")

Lx, Nx = 2 * np.pi, 64
Lt, Nt = 10.0, 1000

def mode(k, c, gam, T, kind='cos'):
    """
    Exact solution for u'' + gam*u' + c^2*k^2*u = 0.
    kind='cos': u(0)=1, u'(0)=0
    kind='sin': u(0)=0, u'(0)=1
    """
    wd = np.sqrt(complex(c**2 * k**2 - gam**2 / 4))
    sinc = np.where(abs(wd * T) < 1e-9, T, np.sin(wd * T) / np.where(wd == 0, 1, wd))
    if kind == 'cos':
        return np.real(np.exp(-gam * T / 2) * (np.cos(wd * T) + gam / 2 * sinc))
    return np.real(np.exp(-gam * T / 2) * sinc)
    
def u_exact(X, T):
    return mode(1, 1.0, 0.8, T, kind='sin') * np.cos(X)

solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    initial_condition=lambda X: np.zeros_like(X),
    initial_velocity=lambda X: np.cos(X),
    plot=True
)

solver.solve()

n_test = 4
# FIX: Added norm='absolute' to prevent division by zero when the exact solution is 0
test_set = [
    solver.test(
        u_exact=u_exact, 
        t_eval=i * Lt / n_test, 
        threshold=1e-6, 
        component='abs',
        norm='absolute'
    )
    for i in range(n_test + 1)
]

### Coefficient a=2 in front of time derivatives + nonlinear term

$$2\,u_{tt} + 1.4\,u_t = 2\,u_{xx} - 2\sin u + 2f$$

Tests scaling of the leading second-order time derivative ($a = 2$) combined with damping, a nonlinear $\sin u$ term, and a manufactured source $f$ constructed so that $u^\star(x,t) = e^{-t}\big(\sin x + \tfrac{1}{2}\cos 2x\big)$ is the exact solution.

In [ ]:
# ==============================================================================
# Example 4: Coefficient a=2 in front of time derivatives + nonlinear term
# Manufactured Solution (MMS)
# Equation: 2*u_tt + 1.4*u_t = 2*u_xx - 2*sin(u) + 2*f
# ==============================================================================
print("\n--- Example 4: Manufactured solution with nonlinear term ---")
ust = exp(-t) * (sin(x) + cos(2 * x) / 2)

# Define the source term 'f' to make 'ust' the exact solution
f_expr = diff(ust, t, 2) + Rational(7, 10) * diff(ust, t) - diff(ust, x, 2) + sin(ust)

eq = Eq(2 * diff(u, t, 2) + Rational(14, 10) * diff(u, t) - 2 * diff(u, x, 2) + 2 * sin(u) - 2 * f_expr, 0)
solver = PDESolver(eq, time_scheme="RK4")

# Exact solution in pure NumPy (no lambdify / dynamic module)
def u_exact(X, T):
    return np.exp(-T) * (np.sin(X) + np.cos(2 * X) / 2)

Lx, Nx = 2 * np.pi, 64
Lt, Nt = 2.0, 1000

solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    initial_condition=lambda X: np.sin(X) + np.cos(2 * X) / 2,
    initial_velocity=lambda X: -(np.sin(X) + np.cos(2 * X) / 2),
    plot=True,
)
solver.solve()

n_test = 4
# 2. Pass the correct u4_exact function here
test_set = [
    solver.test(
        u_exact=u_exact, 
        t_eval=i * Lt / n_test, 
        threshold=1e-4, 
        component='real', 
        norm='absolute'
    )
    for i in range(n_test + 1)
]

### Fractional damped wave with pseudo-differential operator

$$u_{tt} + \gamma\,u_t = \psi\mathrm{Op}\big(-|\xi|^{3/2}, u\big), \qquad \gamma = 1.0$$

Tests a second-order-in-time equation with linear damping and a fractional spatial operator $-(-\Delta)^{3/4}$ via `psiOp` on $[0, 2\pi)$ ($N_x = 64$, $N_t = 1000$). The initial profile $u(x,0) = e^{\cos x - 1}$ is evolved and checked against the exact Bessel–Fourier series at positive times.

In [ ]:
# Define symbols and the equation
t, x, xi = symbols('t x xi', real=True)
U = Function('u')
from scipy.special import ive 

gam = 1.0
p = 1.5
eq = Eq(
    diff(U(t, x), t, 2) + gam * diff(U(t, x), t),
    psiOp(-Abs(xi)**Rational(3, 2), U(t, x))
)
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Exact solution constructed from a sum of Fourier modes
def modefac(w2, gam, T):
    wd = np.sqrt(w2 - gam**2 / 4 + 0j)
    Sx = np.where(np.abs(wd * T) < 1e-9, T, np.sin(wd * T) / np.where(wd == 0, 1, wd))
    return np.real(np.exp(-gam * T / 2) * (np.cos(wd * T) + gam / 2 * Sx))

def u_exact(X, T):
    K = 40
    k = np.arange(-K, K + 1)[:, None]
    c_k = ive(np.abs(k), 1.0)
    return np.real(np.sum(
        c_k * modefac(np.abs(k)**p, gam, T) * np.exp(1j * k * np.asarray(X)[None, :]),
        axis=0
    ))

Lx, Nx = 2 * np.pi, 64
Lt, Nt = 2.0, 1000

solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    initial_condition=lambda X: u_exact(X, 0.0),
    initial_velocity=lambda X: np.zeros_like(X),
    plot=True,
)
solver.solve()

n_test = 6
# Test against exact solution at 6 equally spaced time intervals (skipping t=0)
test_set = [
    solver.test(
        u_exact=u_exact, 
        t_eval=i * Lt / n_test, 
        threshold=1e-5, 
        component='real', 
        norm='relative'
    )
    for i in range(1, n_test + 1)
]

## Equation depending on spatial variables

$$u_{tt} = -\psi\mathrm{Op}(\xi^2 + x^2 + 0.001, u)$$

Tests a second-order wave-type equation with a spatially dependent `psiOp` symbol $\xi^2 + x^2 + 0.001$ on $L_x = 20$ ($N_x = 256$) initialized with the $n = 5$ Hermite eigenfunction $\psi_5(x) = H_5(x)e^{-x^2/2}$ and zero initial velocity. With $\omega = \sqrt{2n + 1 + 0.001}$, the exact standing oscillation is $u(x,t) = \cos(\omega t)\,\psi_5(x)$.

In [ ]:
# Symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

from numpy.polynomial.hermite import hermval

n = 5
Lx, Nx, Lt, Nt = 20.0, 256, 2.0, 400
omega = np.sqrt(2 * n + 1 + 0.001)
hermite = lambda x: hermval(x, [0] * n + [1]) * np.exp(-x**2 / 2)

p_expr = xi**2 + x**2 + 0.001
equation = Eq(diff(u(t, x), t, t), -psiOp(p_expr, u(t, x)))
solver = PDESolver(equation)
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
             boundary_condition='periodic',
             initial_condition=hermite,
             initial_velocity=lambda x: 0 * x,
             n_frames=20, plot=True)
solver.solve()

u_exact = lambda x, t: np.cos(omega * t) * hermite(x)

n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-3, component='real') for i in range(n_test + 1)]


## Schrödinger equation
### With PDE

$$u_t = -i\,u_{xx}, \qquad u(x,0) = e^{-x^2 + ix}$$

Tests free Schrödinger propagation and dispersion of a modulated Gaussian wave packet on $L_x = 20$ ($N_x = 1024$, $N_t = 500$) using `RK4` and dealiasing ratio $1/2$, compared against $u(x,t) = \frac{1}{\sqrt{1 - 4it}}e^{i(x+t)}\exp\!\left(-\frac{(x+2t)^2}{1 - 4it}\right)$.

In [ ]:
# Define the symbols and the 1D Schrödinger equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

eq = Eq(diff(u, t), -I * diff(u, x, x))

# Create the solver with RK4 time scheme
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=1/2)
Lt=2.0
# Configure the domain and initial condition: modulated Gaussian packet
solver.setup(
    Lx=20, Nx=1024, Lt=Lt, Nt=500,
    initial_condition=lambda x: np.exp(-x**2) * np.exp(1j * x)
)

# Solve
solver.solve()

# Exact solution (one of the two propagated packets, here moving to the left)
def u_exact(x, t):
    return 1 / np.sqrt(1 - 4j * t) * np.exp(1j * (x + t)) * np.exp(-((x + 2*t)**2) / (1 - 4j * t))

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=0.5, component='real') for i in range(n_test + 1)]

### With psiOp

$$u_t = -i\,\psi\mathrm{Op}(-\xi^2, u), \qquad u(x,0) = e^{-x^2 + ix}$$

Solves the same free Schrödinger wave-packet problem as above by representing $-u_{xx}$ via `psiOp(-xi**2, u)` with the default solver configuration on $L_x = 20$, $N_x = 1024$.

In [ ]:
# Define the symbols and the 1D Schrödinger equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

eq = Eq(diff(u, t), -I * psiOp(-xi**2, u))

# Create the solver with RK4 time scheme
solver = PDESolver(eq)
Lt=2.0
# Configure the domain and initial condition: modulated Gaussian packet
solver.setup(
    Lx=20, Nx=1024, Lt=Lt, Nt=500,
    initial_condition=lambda x: np.exp(-x**2) * np.exp(1j * x)
)

# Solve
solver.solve()

# Exact solution (one of the two propagated packets, here moving to the left)
def u_exact(x, t):
    return 1 / np.sqrt(1 - 4j * t) * np.exp(1j * (x + t)) * np.exp(-((x + 2*t)**2) / (1 - 4j * t))

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=0.5, component='real') for i in range(n_test + 1)]

### Nonlinear

$$i\,u_t + \tfrac12\,u_{xx} + |u|^2 u = 0$$

Moving bright soliton:

$$u(x,t) = \eta\,\mathrm{sech}\big(\eta(x - vt)\big)\,e^{\,i\left[v x + \frac12(\eta^2 - v^2)t\right]}$$

Points of interest: the coefficient of $u_t$ is $i$, the field is complex (real and imaginary parts are tested separately), and the nonlinearity is cubic. The automatic dealiasing ratio cannot see through `Abs`, so $2/(p+1) = 1/2$ is passed explicitly.

In [ ]:
# Define the symbols and the NLS equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(I * diff(u, t) + diff(u, x, x) / 2 + Abs(u)**2 * u, 0)   # i u_t + u_xx/2 + |u|^2 u = 0

# Soliton parameters
eta = 1.0   # Amplitude and inverse width
v = 0.5     # Velocity

# Exact solution
def u_exact(x, t):
    return eta / np.cosh(eta * (x - v * t)) * np.exp(1j * (v * x + 0.5 * (eta**2 - v**2) * t))

# Initial condition (complex)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (cubic nonlinearity: 1/2 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=1/2)

# Simulation parameters
Lx = 40
Nx = 512
Lt = 5.0
Nt = 1000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: real and imaginary parts
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-6, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-6, component='imag') for i in range(n_test + 1)]

### Akhmediev breather (modulation instability)

$$i\,u_t + \tfrac12\,u_{xx} + |u|^2 u = 0$$

Exact solution, periodic in $x$ and localized in time (one growth–decay cycle of the modulational instability of a plane wave), for $0<a<\tfrac12$:

$$u(x,t) = e^{it}\,\frac{(1-4a)\cosh bt + i\,b\sinh bt + \sqrt{2a}\cos\Omega x}{\sqrt{2a}\cos\Omega x - \cosh bt},\qquad b = \sqrt{8a(1-2a)},\ \ \Omega = 2\sqrt{1-2a}$$

The domain is exactly one period $2\pi/\Omega$. The solver starts at $t=0$, which is mapped to the breather time $t_\mathrm{start} = -3$, before the instability has developed. The problem is unstable by nature, so tiny errors are amplified by about $e^{b\,L_t}$.

In [ ]:
# Define the symbols and the NLS equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(I * diff(u, t) + diff(u, x, x) / 2 + Abs(u)**2 * u, 0)   # i u_t + u_xx/2 + |u|^2 u = 0

# Breather parameters
a = 0.25                                  # 0 < a < 1/2 (controls the modulation period)
b = np.sqrt(8 * a * (1 - 2 * a))          # Growth rate
Omega = 2 * np.sqrt(1 - 2 * a)            # Modulation wavenumber
t_start = -3.0                            # Solver time 0 <-> breather time t_start

# Exact solution
def u_exact(x, t):
    tau = t + t_start
    num = (1 - 4 * a) * np.cosh(b * tau) + 1j * b * np.sinh(b * tau) + np.sqrt(2 * a) * np.cos(Omega * x)
    den = np.sqrt(2 * a) * np.cos(Omega * x) - np.cosh(b * tau)
    return np.exp(1j * tau) * num / den

# Initial condition (complex)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (cubic nonlinearity: 1/2 rule) <= it should be checked seriously
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=1/2)

# Simulation parameters
Lx = 2 * np.pi / Omega     # Exactly one spatial period
Nx = 256
Lt = 6.0
Nt = 3000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: real and imaginary parts
# n_test = 5 on purpose: with t_start = -3, a checkpoint at t = 3 would fall on tau = 0, where Im(u) = 0
# exactly and the relative error of the imaginary part is undefined (0/0)
n_test = 5
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-7, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-7, component='imag') for i in range(n_test + 1)]

### Gaussian wavepacket

$$i\,u_t + \tfrac12\,u_{xx} = 0 \qquad(\hbar = m = 1)$$

A Gaussian packet of width $\sigma$ and mean momentum $k_0$ spreads and drifts:

$$u(x,t) = \frac{1}{\sqrt{1 + it/\sigma^2}}\,\exp\!\left(-\frac{(x - k_0 t)^2}{2\sigma^2(1 + it/\sigma^2)} + i\Big(k_0 x - \frac{k_0^2 t}{2}\Big)\right),\qquad u(x,0) = e^{-x^2/2\sigma^2 + i k_0 x}$$

The problem is linear, so the exponential integrator is exact in time: any visible error would point to the dispersion relation $\omega = k^2/2$ or to the handling of the complex coefficient $i$ in front of $u_t$. The packet is localized, hence the large periodic domain: it must stay away from the boundaries for the whole run (the width grows to about $6.4$ at $t = 8$).

In [ ]:
# Define the symbols and the free Schrodinger equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(I * diff(u, t) + diff(u, x, x) / 2, 0)   # i u_t + u_xx/2 = 0

# Wavepacket parameters
sigma = 1.3   # Initial width
k0 = 0.8      # Mean momentum

# Exact solution
def u_exact(x, t):
    d = 1 + 1j * t / sigma**2
    return d**(-0.5) * np.exp(-(x - k0 * t)**2 / (2 * sigma**2 * d) + 1j * (k0 * x - k0**2 * t / 2))

# Initial condition (complex)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 120
Nx = 512
Lt = 8.0
Nt = 400

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: real and imaginary parts
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='imag') for i in range(n_test + 1)]

### Free Schrödinger equation on a circle: quantum revivals (Talbot effect)

$$i\,u_t + \tfrac12\,u_{xx} = 0,\qquad x\in[-\pi,\pi)\ \text{periodic},\qquad u(x,0) = e^{\cos x - 1}$$

With $e^{\cos x - 1} = \sum_k \tilde I_k(1)\,e^{ikx}$ (Jacobi–Anger, $\tilde I_k$ = exponentially scaled modified Bessel function), the exact solution is the Fourier series

$$u(x,t) = \sum_{k\in\mathbb Z} \tilde I_{|k|}(1)\;e^{\,i(kx - k^2 t/2)}$$

Because $k^2$ is an integer, the dynamics is exactly periodic in time: **full revival** $u(x,4\pi) = u(x,0)$, and **half-period shift** $u(x,2\pi) = u(x+\pi,0)$, with fractional revivals in between (the "quantum carpet"). Between revivals the solution is a complicated, highly oscillating wave, a very sensitive test of the dispersion relation over a long time ($L_t = 4\pi$).

In [ ]:
# Define the symbols and the free Schrodinger equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)
from scipy.special import ive 

eq = Eq(I * diff(u, t) + diff(u, x, x) / 2, 0)   # i u_t + u_xx/2 = 0

# Exact solution: Fourier series with coefficients ive(|k|, 1) (exp(cos x - 1) = sum_k ive(|k|, 1) e^{ikx})
def u_exact(x, t):
    x = np.asarray(x)[None, :]
    K = 40
    k = np.arange(-K, K + 1)[:, None]
    c_k = ive(np.abs(k), 1.0)
    return np.sum(c_k * np.exp(1j * (k * x - 0.5 * k**2 * t)), axis=0)

# Initial condition
initial_condition = lambda x: np.exp(np.cos(x) - 1)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi
Nx = 64
Lt = 4 * np.pi        # Full revival time
Nt = 2000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: n_test = 8 puts checkpoints at t = pi/2, pi, ..., 2 pi (half-period shift), ..., 4 pi (revival)
# Im(u) vanishes at the revival times, so it is tested in absolute norm
n_test = 8
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, norm='absolute', threshold=1e-10, component='imag') for i in range(n_test + 1)]

### Schrödinger equation with a harmonic potential: coherent state

$$i\,u_t = -\tfrac12\,u_{xx} + \tfrac12\,x^2\,u$$

A coherent state (displaced ground state) oscillates without deformation with the classical period $2\pi$:

$$u(x,t) = \pi^{-1/4}\exp\!\Big(-\tfrac12\big(x - x_0\cos t\big)^2 - i\,x\,x_0\sin t - \tfrac{i t}{2} + \tfrac{i}{4}x_0^2\sin 2t\Big)$$

**This example needs the variable-coefficient fix in `_parse_equation`.** Without it, the term $-\tfrac12 x^2 u$ is silently read as $1\cdot u$ (the coefficient is lost: `term.as_coefficients_dict().get(self.u, 1)` returns the default `1` because the dictionary key is the monomial `x**2*u`, not `u`) and the solution is wrong, with an error of order 1 and no warning. With the fix, such terms are evaluated pointwise, like the nonlinear terms.

The potential is treated explicitly by the integrator ($|V|\,\Delta t$ must stay small), hence the fine time step; the error decreases like $\Delta t^4$ (1.7e-8, 1e-9, 6e-11, 4e-12 for $N_t$ = 1000, 2000, 4000, 8000).

In [ ]:
# Define the symbols and the Schrodinger equation with a harmonic potential
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(I * diff(u, t) + diff(u, x, x) / 2 - x**2 * u / 2, 0)   # i u_t = -u_xx/2 + x^2 u/2

# Coherent state parameters
x0 = 1.5   # Initial displacement

# Exact solution
def u_exact(x, t):
    return np.pi**(-0.25) * np.exp(-(x - x0 * np.cos(t))**2 / 2 - 1j * x * x0 * np.sin(t)
                                   - 0.5j * t + 0.25j * x0**2 * np.sin(2 * t))

# Initial condition (real Gaussian displaced by x0)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 18          # The packet stays within |x| < 6: negligible tails at the boundaries
Nx = 128
Lt = 2 * np.pi   # One classical period
Nt = 4000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing in absolute norm: Re(u) = 0 at t = pi (and Im(u) = 0 at t = 0), so relative errors are undefined there
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, norm='absolute', threshold=1e-9, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, norm='absolute', threshold=1e-9, component='imag') for i in range(n_test + 1)]

### Focusing NLS: Kuznetsov–Ma breather

$$i\,u_t + \tfrac12\,u_{xx} + |u|^2 u = 0$$

Exact solution on a plane-wave background $e^{it}$, **localized in $x$ and periodic in time**, for $a > \tfrac12$:

$$u(x,t) = e^{it}\,\frac{(1-4a)\cos\beta t - i\,\beta\sin\beta t + \sqrt{2a}\cosh\kappa x}{\sqrt{2a}\cosh\kappa x - \cos\beta t},\qquad \kappa = 2\sqrt{2a-1},\ \ \beta = \sqrt{8a(2a-1)}$$

(the companion of the Akhmediev breather, which is periodic in $x$ and localized in $t$). The peak amplitude at $t=0$ is $|u(0,0)| = 3.83$ for $a=1$, on a unit background. The background plane wave is modulationally unstable, so numerical noise is amplified: the test is demanding. The approach to the background is exponential ($e^{-\kappa|x|}$), which makes the periodic domain exact to machine precision.

The peak is thin (the singularities of $u$ sit at $|\mathrm{Im}\,x| \approx 0.4$, so the spectrum decays only like $e^{-0.4k}$): with the $1/2$ dealiasing rule $N_x = 1024$ is required (relative error 1e-3 with $N_x = 512$). With $N_x = 1024$ the remaining error is the time discretization, of order $\Delta t^4$ (9e-8, 6e-9, 5e-10 for $N_t$ = 4000, 8000, 16000).

In [ ]:
# Define the symbols and the NLS equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(I * diff(u, t) + diff(u, x, x) / 2 + Abs(u)**2 * u, 0)   # i u_t + u_xx/2 + |u|^2 u = 0

# Breather parameters
a = 1.0                                   # a > 1/2
kappa = 2 * np.sqrt(2 * a - 1)            # Spatial localization
beta = np.sqrt(8 * a * (2 * a - 1))       # Temporal frequency (period 2 pi / beta)

# Exact solution
def u_exact(x, t):
    num = (1 - 4 * a) * np.cos(beta * t) - 1j * beta * np.sin(beta * t) + np.sqrt(2 * a) * np.cosh(kappa * x)
    den = np.sqrt(2 * a) * np.cosh(kappa * x) - np.cos(beta * t)
    return np.exp(1j * t) * num / den

# Initial condition (complex)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (cubic nonlinearity: 1/2 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=1/2)

# Simulation parameters
Lx = 24                       # Tails e^{-kappa Lx/2} ~ 4e-11
Nx = 1024                     # The thin peak needs it (error 1e-3 with Nx = 512)
Lt = 2 * 2 * np.pi / beta     # Two periods of the breather
Nt = 8000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing (Im(u) = 0 at t = 0: absolute norm for the imaginary part)
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=5e-7, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, norm='absolute', threshold=5e-7, component='imag') for i in range(n_test + 1)]

### Focusing NLS: a rogue wave from a whisper

$$i\,u_t + \tfrac12\,u_{xx} + |u|^2 u = 0$$

A plane wave of unit amplitude is unstable (modulational instability). The Akhmediev breather, exact and periodic in $x$, describes the complete life of the instability: a **0.2 % modulation** of the background grows exponentially, focuses at $t_0 = 0$ into a giant wave of amplitude $1 + 2\sqrt{2a} = 2.90$ (the Peregrine rogue wave has 3 and is the limit $a\to\tfrac12$), then decays back to the plane wave:

$$u(x,t) = e^{it}\,\frac{(1-4a)\cosh bt + i\,b\sinh bt + \sqrt{2a}\cos\Omega x}{\sqrt{2a}\cos\Omega x - \cosh bt},\qquad b = \sqrt{8a(1-2a)},\ \ \Omega = 2\sqrt{1-2a}$$

Here $a = 0.45$ ($b = 0.6$, $L_x = 2\pi/\Omega = 9.93$) and the solver starts at breather time $-10$, when the modulation is only $1.9\times10^{-3}$ peak to peak. It is a demanding numerical test: the background is unstable, so every numerical error is amplified like $e^{bt}$ (about $10^{5}$ over the run). Nevertheless the relative error stays below $10^{-7}$ up to the peak and reaches $2.6\times10^{-7}$ at the end ($\propto\Delta t^4$: 5e-5, 4e-6, 3e-7 for $N_t$ = 5000, 10000, 20000).

In [ ]:
# Define the symbols and the NLS equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(I * diff(u, t) + diff(u, x, x) / 2 + Abs(u)**2 * u, 0)   # i u_t + u_xx/2 + |u|^2 u = 0

# Breather parameters
a = 0.45                                  # Close to the Peregrine limit a = 1/2
b = np.sqrt(8 * a * (1 - 2 * a))          # Growth rate of the instability
Omega = 2 * np.sqrt(1 - 2 * a)            # Modulation wavenumber
t_start = -10.0                           # Solver time 0 <-> breather time -10 (modulation ~ 2e-3)

# Exact solution
def u_exact(x, t):
    tau = t + t_start
    num = (1 - 4 * a) * np.cosh(b * tau) + 1j * b * np.sinh(b * tau) + np.sqrt(2 * a) * np.cos(Omega * x)
    den = np.sqrt(2 * a) * np.cos(Omega * x) - np.cosh(b * tau)
    return np.exp(1j * tau) * num / den

# Initial condition: a nearly flat plane wave (complex)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (cubic nonlinearity: 1/2 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=1/2)

# Simulation parameters
Lx = 2 * np.pi / Omega     # Exactly one spatial period
Nx = 512
Lt = 20.0                  # From breather time -10 to +10
Nt = 20000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: real part (relative norm) and imaginary part (absolute norm).
# n_test = 6 puts a checkpoint exactly at the peak t = 10, where Im(u) = 0: its relative error would be 0/0
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=2e-6, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, norm='absolute', threshold=5e-5, component='imag') for i in range(n_test + 1)]

#### Visualization: the rise and fall of the rogue wave

In [ ]:
frames = np.array(solver.frames)
times = np.arange(len(frames)) * max(1, Nt // solver.n_frames) * (Lt / Nt)
X = solver.X
exact = np.array([u_exact(X, tt) for tt in times])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

# Space-time map of |u|: the instability focuses at t = 10 (breather time 0)
im = axes[0].pcolormesh(X, times, np.abs(frames), shading='auto', cmap='inferno')
axes[0].set_xlabel('x'); axes[0].set_ylabel('t'); axes[0].set_title('|u(x, t)|')
fig.colorbar(im, ax=axes[0])

# Peak amplitude versus time: numerical (line) vs exact (dots)
axes[1].plot(times, np.abs(frames).max(axis=1), 'tab:red', lw=2, label='numerical')
axes[1].plot(times[::3], np.abs(exact).max(axis=1)[::3], 'k.', ms=4, label='exact')
axes[1].set_xlabel('t'); axes[1].set_ylabel('max |u|'); axes[1].set_title('amplitude: 1 -> 2.90 -> 1')
axes[1].legend()

# Profiles around the peak
for ts, color in zip((0.0, 6.0, 9.0, 10.0), ('k', 'tab:blue', 'tab:green', 'tab:red')):
    j = np.argmin(np.abs(times - ts))
    axes[2].plot(X, np.abs(frames[j]), color=color, lw=1.5, label=f't = {times[j]:.1f}')
    axes[2].plot(X[::6], np.abs(exact[j])[::6], '.', color=color, ms=3)
axes[2].set_xlabel('x'); axes[2].set_title('|u| profiles: numerical (lines) vs exact (dots)')
axes[2].legend()
plt.tight_layout()
plt.show()

## Klein-Gordon equation
### RK4

$$u_{tt} = u_{xx} - u, \qquad u(x,0) = \cos x,\quad u_t(x,0) = 0$$

Tests the linear Klein–Gordon equation on $L_x = 6\pi$ ($N_x = 1024$, $N_t = 200$) using `RK4` with dealiasing ratio $1/4$, plotting energy and verifying harmonic oscillation at frequency $\sqrt{2}$: $u(x,t) = \cos(\sqrt{2}\,t)\cos x$.

In [ ]:
# Define the symbols and the Klein-Gordon equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

eq = Eq(diff(u, t, t), diff(u, x, x) - u)

# Create the solver with RK4 scheme
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=1/4)
Lt=2.0
# Simulation parameters
solver.setup(
    Lx=6*np.pi, Nx=1024, Lt=Lt, Nt=200,
    initial_condition=lambda x: np.cos(x),
    initial_velocity=lambda x: np.zeros_like(x)
)

# Solving
solver.solve()

solver.plot_energy() #(log=True) 

# Exact solution at t = 2: harmonic oscillation with frequency sqrt(2)
def u_exact(x, t):
    return np.cos(np.sqrt(2) * t) * np.cos(x)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-8, component='real') for i in range(n_test + 1)]


### Default scheme

In [ ]:
# Define the symbols and the Klein-Gordon equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

eq = Eq(diff(u, t, t), diff(u, x, x) - u)

# Create the solver with RK4 scheme
solver = PDESolver(eq, dealiasing_ratio=2/3)
Lt=2.0
# Simulation parameters
solver.setup(
    Lx=6*np.pi, Nx=1024, Lt=Lt, Nt=200,
    initial_condition=lambda x: np.cos(x),
    initial_velocity=lambda x: np.zeros_like(x)
)

# Solving
solver.solve()

solver.plot_energy() #(log=True) 

# Exact solution at t = 2: harmonic oscillation with frequency sqrt(2)
def u_exact(x, t):
    return np.cos(np.sqrt(2) * t) * np.cos(x)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-13, component='real') for i in range(n_test + 1)]


### Periodic cnoidal wave

$$u_{tt} = u_{xx} - u - u^3$$

Exact periodic traveling wave $u = A\,\mathrm{cn}\big(\Omega(\kappa x - \omega t)\,\big|\,m\big)$ with

$$A^2 = \frac{2m}{1-2m},\qquad \Omega^2 = \frac{1}{s\,(1-2m)},\qquad \omega^2 = \kappa^2 + s,\qquad 0<m<\tfrac12$$

($s = \omega^2-\kappa^2 > 0$, free). The domain is exactly one spatial period $4K(m)/(\Omega\kappa)$. Second order in time, cubic nonlinearity (dealiasing ratio $1/2$), non-zero initial velocity $u_t(x,0) = A\Omega\omega\,\mathrm{sn}\,\mathrm{dn}$.

In [ ]:
# Define the symbols and the nonlinear Klein-Gordon equation
t, x, kx, xi, omega_ = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(diff(u, t, t) - diff(u, x, x) + u + u**3, 0)   # u_tt = u_xx - u - u^3

# Cnoidal wave parameters
m = 0.3                                  # Elliptic parameter (0 < m < 1/2)
kappa = 1.0
s = 1.0                                  # s = omega^2 - kappa^2
A = np.sqrt(2 * m / (1 - 2 * m))         # Amplitude
Omega = 1 / np.sqrt(s * (1 - 2 * m))
omega = np.sqrt(kappa**2 + s)            # Temporal frequency

# Exact solution
def u_exact(x, t):
    sn, cn, dn, ph = ellipj(Omega * (kappa * x - omega * t), m)
    return A * cn

# Initial conditions
initial_condition = lambda x: u_exact(x, 0.0)
def initial_velocity(x):
    sn, cn, dn, ph = ellipj(Omega * kappa * x, m)
    return A * Omega * omega * sn * dn

# Create the solver (cubic nonlinearity: 1/2 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=1/2)

# Simulation parameters
Lx = 4 * ellipk(m) / (Omega * kappa)     # Exactly one spatial period
Nx = 128
Lt = 6.0                                 # About two temporal periods
Nt = 2000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
             initial_condition=initial_condition,
             initial_velocity=initial_velocity)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-8, component='real') for i in range(n_test + 1)]

## KdV equation
### To the left

$$u_t + 6\,u\,u_x - u_{xxx} = 0$$

Tests a left-propagating soliton on $L_x = 40$ ($N_x = 1024$, $L_t = 3.0$, $N_t = 1000$) using `RK4` and $2/3$ dealiasing for speed/amplitude parameter $c = 0.5$, $x_0 = 0$:

$$u(x,t) = -\frac{c}{2}\operatorname{sech}^2\!\left(\frac{\sqrt{c}}{2}(x + ct - x_0)\right)$$

In [ ]:
# Define the symbols and the KdV equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)


eq = Eq(diff(u, t) + 6 * u * diff(u, x) - diff(u, x, x, x), 0)   # u_t + 6 u u_x + u_xxx = 0

# Soliton parameters
c = 0.5   # Speed and amplitude of the soliton
x0 = 0.0  # Initial position

# Initial condition: centered soliton
initial_condition = lambda x: -c / 2 / np.cosh(np.sqrt(c) / 2 * (x - x0))**2


# Create the solver with periodic boundary conditions and adapted dealiasing
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 40
Nx = 1024
Lt = 3.0
Nt = 1000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Exact solution at t = Lt
def u_exact(x, t):
    return -c / 2 / np.cosh(np.sqrt(c) / 2 * (x + c * t - x0))**2

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=0.5, component='real') for i in range(n_test + 1)]


### To the right

$$u_t + 6\,u\,u_x + u_{xxx} = 0$$

Tests a right-propagating soliton on $L_x = 40$ ($N_x = 1024$, $L_t = 3.0$, $N_t = 1000$) using the default time integrator and $2/3$ dealiasing for $c = 0.5$, $x_0 = 0$:

$$u(x,t) = \frac{c}{2}\operatorname{sech}^2\!\left(\frac{\sqrt{c}}{2}(x - ct - x_0)\right)$$

In [ ]:
# Define the symbols and the KdV equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

eq = Eq(diff(u, t) + 6 * u * diff(u, x) + diff(u, x, x, x), 0)    # u_t + 6 u u_x + u_xxx = 0


# Soliton parameters
c = 0.5   # Speed and amplitude of the soliton
x0 = 0.0  # Initial position

# Initial condition: centered soliton
initial_condition = lambda x: c / 2 / np.cosh(np.sqrt(c) / 2 * (x - x0))**2


# Create the solver with periodic boundary conditions and adapted dealiasing
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Lx = 40
Nx = 1024
Lt = 3.0
Nt = 1000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Exact solution at t = Lt
def u_exact(x, t):
    return c / 2 / np.cosh(np.sqrt(c) / 2 * (x - c * t - x0))**2

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=0.5, component='real') for i in range(n_test + 1)]


### One soliton
$$u_t + 6\,u\,u_x + u_{xxx} = 0$$

It tests a third derivative in the linear part together with a quadratic nonlinearity.

* One soliton: $u = \dfrac{c}{2}\,\mathrm{sech}^2\!\Big(\dfrac{\sqrt c}{2}(x - ct - x_0)\Big)$.

In [ ]:
# Define the symbols and the KdV equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(diff(u, t) + 6 * u * diff(u, x) + diff(u, x, x, x), 0)   # u_t + 6 u u_x + u_xxx = 0

# Soliton parameters
c = 1.0    # Speed (the amplitude is c/2)
x0 = -5.0  # Initial position

# Exact solution
def u_exact(x, t):
    return c / 2 / np.cosh(np.sqrt(c) / 2 * (x - c * t - x0))**2

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (quadratic nonlinearity: 2/3 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 100
Nx = 512
Lt = 10.0
Nt = 2000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-8, component='real') for i in range(n_test + 1)]

### Two-soliton collision

$$u_t + 6\,u\,u_x + u_{xxx} = 0$$

It tests a third derivative in the linear part together with a quadratic nonlinearity.
* Two solitons: $u = 2\,\partial_x^2 \log\tau$ with $\tau = 1 + e^{\eta_1} + e^{\eta_2} + A_{12}\,e^{\eta_1+\eta_2}$, $\eta_i = k_i(x - x_i) - k_i^3 t$ and $A_{12} = \big(\tfrac{k_1-k_2}{k_1+k_2}\big)^2$ (speeds $k_i^2$).

The solitons are localized, so the periodic domain just has to be large compared with their width.

In [ ]:
# Two-soliton parameters
k1, k2 = 1.5, 1.0
x1, x2 = -25.0, -5.0
A12 = ((k1 - k2) / (k1 + k2))**2


# Exact solution (2-soliton)
def u_exact(x, t):
    eta1 = k1 * (x - x1) - k1**3 * t
    eta2 = k2 * (x - x2) - k2**3 * t
    
    e1 = np.exp(eta1)
    e2 = np.exp(eta2)
    e12 = A12 * np.exp(eta1 + eta2)
    
    tau = 1.0 + e1 + e2 + e12
    tau_x = k1 * e1 + k2 * e2 + (k1 + k2) * e12
    tau_xx = k1**2 * e1 + k2**2 * e2 + (k1 + k2)**2 * e12
    
    return 2.0 * (tau * tau_xx - tau_x**2) / tau**2

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 100
Nx = 512
Lt = 20.0
Nt = 8000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-6, component='real') for i in range(n_test + 1)]

### Periodic cnoidal wave

$$u_t + 6\,u\,u_x + u_{xxx} = 0$$

Exact periodic traveling wave (no localized tails, hence no wrap-around error):

$$u(x,t) = \beta + 2 m\kappa^2\,\mathrm{cn}^2\!\big(\kappa(x - ct)\,\big|\,m\big),\qquad c = 6\beta + 4\kappa^2(2m-1)$$

with elliptic parameter $0<m<1$ and spatial period $2K(m)/\kappa$. As $m\to 1$ it tends to a train of solitons. Here $m = 0.9$, $\beta = 0.5$.

In [ ]:
# Define the symbols and the KdV equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(diff(u, t) + 6 * u * diff(u, x) + diff(u, x, x, x), 0)   # u_t + 6 u u_x + u_xxx = 0

# Cnoidal wave parameters
m = 0.9        # Elliptic parameter
kappa = 1.0    # Wavenumber scale
beta = 0.5     # Mean level
c = 6 * beta + 4 * kappa**2 * (2 * m - 1)   # Speed

# Exact solution
def u_exact(x, t):
    sn, cn, dn, ph = ellipj(kappa * (x - c * t), m)
    return beta + 2 * m * kappa**2 * cn**2

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (quadratic nonlinearity: 2/3 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * ellipk(m) / kappa    # Exactly one spatial period
Nx = 128
Lt = 2.0
Nt = 4000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-8, component='real') for i in range(n_test + 1)]

### KdV: a single smooth hump splits into two solitons

$$u_t + 6\,u\,u_x + u_{xxx} = 0,\qquad u(x,0) = 6\,\mathrm{sech}^2 x$$

The initial condition is not a soliton, just a smooth bump. Yet the evolution is known in closed form (inverse scattering: $-u_0 = -6\,\mathrm{sech}^2x$ is a *reflectionless* Pöschl–Teller potential with exactly two bound states, $\kappa = 1$ and $\kappa = 2$). The hump separates **exactly**, with no radiation, into two solitons of amplitudes $2\kappa^2$ and speeds $4\kappa^2$: a tall one (amplitude 8, speed 16) that outruns a small one (amplitude 2, speed 4).

$$u(x,t) = \frac{12\,\big[3 + 4\cosh(2x - 8t) + \cosh(4x - 64t)\big]}{\big[3\cosh(x - 28t) + \cosh(3x - 36t)\big]^2}$$

At $t=0$ this reduces to $6\,\mathrm{sech}^2x$. The small displacement of the slow soliton with respect to $x = 4t$ is the phase shift due to the initial overlap. The test checks the whole separation process, with an error decreasing like $\Delta t^4$ (3.4e-6, 1.5e-7, 7.0e-9 for $N_t$ = 3000, 6000, 12000).

In [ ]:
# Define the symbols and the KdV equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(diff(u, t) + 6 * u * diff(u, x) + diff(u, x, x, x), 0)   # u_t + 6 u u_x + u_xxx = 0

# Exact solution: the hump 6 sech^2(x) splits into two solitons (amplitude 8 at speed 16, amplitude 2 at speed 4)
def u_exact(x, t):
    num = 3 + 4 * np.cosh(2 * x - 8 * t) + np.cosh(4 * x - 64 * t)
    den = 3 * np.cosh(x - 28 * t) + np.cosh(3 * x - 36 * t)
    return 12 * num / den**2

# Initial condition: a single smooth hump
initial_condition = lambda x: 6 / np.cosh(x)**2

# Create the solver (quadratic nonlinearity: 2/3 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 60       # Large enough for the fast soliton (x = 16 t) to stay far from the boundary
Nx = 1024
Lt = 1.0
Nt = 12000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-7, component='real') for i in range(n_test + 1)]

#### Visualization: space-time map and snapshots (numerical solution vs exact formula)

In [ ]:
frames = np.real(np.array(solver.frames))
times = np.arange(len(frames)) * max(1, Nt // solver.n_frames) * (Lt / Nt)
X = solver.X

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Space-time map: one hump in, two solitons out (the tall one is faster)
im = axes[0].pcolormesh(X, times, frames, shading='auto', cmap='magma')
axes[0].set_xlim(-6, 24)
axes[0].set_xlabel('x'); axes[0].set_ylabel('t'); axes[0].set_title('u(x, t): one hump becomes two solitons')
fig.colorbar(im, ax=axes[0], label='u')

# Snapshots: numerical solution (lines) and exact formula (dots)
for ts, color in zip((0.0, 0.1, 0.25, 0.5, 1.0), ('k', 'tab:blue', 'tab:green', 'tab:orange', 'tab:red')):
    j = np.argmin(np.abs(times - ts))
    axes[1].plot(X, frames[j], color=color, lw=1.5, label=f't = {times[j]:.2f}')
    axes[1].plot(X[::6], u_exact(X[::6], times[j]), '.', color=color, ms=3)
axes[1].set_xlim(-6, 24)
axes[1].set_xlabel('x'); axes[1].set_title('numerical (lines) vs exact (dots)')
axes[1].legend()
plt.tight_layout()
plt.show()

## Biharmonic equation

$$u_t = -u_{xxxx}, \qquad u(x,0) = \sin x$$

Tests fourth-order spatial dissipation on $[0, 2\pi)$ ($N_x = 1024$, $L_t = 1.0$, $N_t = 200$) with periodic boundary conditions. Since $\partial_x^4 \sin x = \sin x$, the exact solution is $u(x,t) = e^{-t}\sin x$.

In [ ]:
# Definition of the 1D biharmonic equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u') 
u = u_func(t, x)

biharmonic_eq = Eq(diff(u, t), -diff(u, x, x, x, x))

# Creation of the solver with periodic boundary conditions
solver = PDESolver(biharmonic_eq)

# Configuration of the domain and initial condition
Lx = 2 * np.pi
Nx = 1024
Lt = 1.0
Nt = 200
solver.setup(
    Lx=Lx, Nx=Nx,
    Lt=Lt, Nt=Nt,
    initial_condition=lambda x: np.sin(x)
)

# Solving
solver.solve()

# Exact solution
def u_exact(x, t):
    return np.sin(x) * np.exp(-t)

# Automatic testing
n_test = 4
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

## Sine-Gordon equation
### Default scheme

$$u_{tt} = u_{xx} - \sin u$$

Tests a stationary/pulsating sine–Gordon breather ($w = 0.5$, $k = \sqrt{1 - w^2}$) on a wide domain ($L_x = 80$, $N_x = 1024$, $L_t = 4.0$, $N_t = 2000$) using the default time scheme and $2/3$ dealiasing:

$$u(x,t) = 4\arctan\!\left(\frac{k}{w}\frac{\sin(wt)}{\cosh(kx)}\right), \qquad u_t(x,0) = \frac{4k}{\cosh(kx)}$$

Because $u(x,0) \equiv 0$, $t = 0$ is checked with `norm='absolute'` while $t > 0$ uses relative error.

In [ ]:
# Sine-Gordon Breather
t, x = symbols('t x')
u = Function('u')(t, x)
eq = Eq(diff(u, t, t), diff(u, x, x) - sin(u))

w = 0.5; k = np.sqrt(1 - w**2)
def u_exact(x, t): return 4*np.arctan((k/w)*np.sin(w*t)/np.cosh(k*x))
def u_vel(x):      return 4*k/np.cosh(k*x)

solver = PDESolver(eq, dealiasing_ratio=2/3)
# Increased Lx to 80 to push the spatial wrap-around error below 1e-14
solver.setup(Lx=80, Nx=1024, Lt=4.0, Nt=2000, plot=True,
             initial_condition=lambda x: u_exact(x, 0),
             initial_velocity=u_vel)
solver.solve()

n_test = 4
solver.test(u_exact=u_exact, t_eval=0.0, norm='absolute', threshold=1e-12, component='real')  # IC check
test_set = [solver.test(u_exact=u_exact, t_eval=i*4.0/n_test, threshold=5e-3, component='real')
            for i in range(1, n_test + 1)]          # start at i=1: relative norm undefined at t=0

### RK4

$$u_{tt} = u_{xx} - \sin u$$

Runs the same sine–Gordon breather ($w = 0.5$, $L_x = 80$, $N_x = 1024$, $L_t = 4.0$, $N_t = 2000$) using `RK4` and $2/3$ dealiasing, achieving tighter accuracy (`threshold=1e-12`) at positive times than the default scheme.

In [ ]:
# Sine-Gordon Breather
t, x = symbols('t x')
u = Function('u')(t, x)
eq = Eq(diff(u, t, t), diff(u, x, x) - sin(u))

w = 0.5; k = np.sqrt(1 - w**2)
def u_exact(x, t): return 4*np.arctan((k/w)*np.sin(w*t)/np.cosh(k*x))
def u_vel(x):      return 4*k/np.cosh(k*x)

solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)
# Increased Lx to 80 to push the spatial wrap-around error below 1e-14
solver.setup(Lx=80, Nx=1024, Lt=4.0, Nt=2000, plot=True,
             initial_condition=lambda x: u_exact(x, 0),
             initial_velocity=u_vel)
solver.solve()

n_test = 4
solver.test(u_exact=u_exact, t_eval=0.0, norm='absolute', threshold=1e-12, component='real')  # IC check
test_set = [solver.test(u_exact=u_exact, t_eval=i*4.0/n_test, threshold=1e-12, component='real')
            for i in range(1, n_test + 1)]          # start at i=1: relative norm undefined at t=0

### Kink–antikink collision

$$u_{tt} = u_{xx} - \sin u$$

$$u(x,t) = 4\arctan\!\left(\frac{\sinh(\gamma v t)}{v\cosh(\gamma x)}\right),\qquad \gamma = \frac{1}{\sqrt{1-v^2}}$$

Initial data: $u(x,0) = 0$ and $u_t(x,0) = 4\gamma/\cosh(\gamma x)$. A plateau at $2\pi$ grows in the middle while the two fronts move apart. Second order in time, with a non-polynomial nonlinearity (so the dealiasing ratio is given explicitly).

Since $u(x,0)\equiv 0$, the relative error is undefined at $t=0$: the initial condition is checked in absolute norm and the relative tests start at $i=1$.

In [ ]:
# Define the symbols and the sine-Gordon equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(diff(u, t, t) - diff(u, x, x) + sin(u), 0)   # u_tt = u_xx - sin(u)

# Parameters
v = 0.5
gamma = 1 / np.sqrt(1 - v**2)

# Exact solution
def u_exact(x, t):
    return 4 * np.arctan(np.sinh(gamma * v * t) / (v * np.cosh(gamma * x)))

# Initial conditions
initial_condition = lambda x: u_exact(x, 0.0)
initial_velocity = lambda x: 4 * gamma / np.cosh(gamma * x)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 80
Nx = 1024
Lt = 20.0
Nt = 4000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
             initial_condition=initial_condition,
             initial_velocity=initial_velocity)

# Solve
solver.solve()

# Automatic testing
n_test = 6
solver.test(u_exact=u_exact, t_eval=0.0, norm='absolute', threshold=1e-12, component='real')   # IC check
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-8, component='real') for i in range(1, n_test + 1)]

## Ginzburg–Landau equation

### nonlinear plane wave

$$u_t = u + (1 + i\alpha)\,u_{xx} - (1 + i\beta)\,|u|^2 u$$

Exact solution (Stokes wave):

$$u(x,t) = \sqrt{1-k^2}\;e^{\,i(kx - \omega t)},\qquad \omega = \alpha k^2 + \beta(1-k^2),\quad |k|<1$$

It is stable for $1 + \alpha\beta > 0$ (Benjamin–Feir–Newell criterion). The point of this test is the **complex coefficients** in the linear part and in the nonlinear term, together with $|u|^2u$, in a diffusive (non-conservative) setting.

In [ ]:
# Define the symbols and the complex Ginzburg-Landau equation
t, x, kx, xi, omega_ = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

alpha, beta = 0.5, 0.3     # 1 + alpha*beta > 0: stable plane wave
eq = Eq(diff(u, t) - u - (1 + I * alpha) * diff(u, x, x) + (1 + I * beta) * Abs(u)**2 * u, 0)
# u_t = u + (1 + i alpha) u_xx - (1 + i beta) |u|^2 u

# Plane wave parameters
k = 0.5                                       # |k| < 1
omega = alpha * k**2 + beta * (1 - k**2)      # Frequency

# Exact solution
def u_exact(x, t):
    return np.sqrt(1 - k**2) * np.exp(1j * (k * x - omega * t))

# Initial condition (complex)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (cubic nonlinearity: 1/2 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=1/2)

# Simulation parameters
Lx = 2 * np.pi / k         # Exactly one wavelength
Nx = 64
Lt = 10.0
Nt = 1000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: real and imaginary parts
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-9, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-9, component='imag') for i in range(n_test + 1)]

## Boussinesq equation

### Soliton

$$u_{tt} = u_{xx} - u_{xxxx} + (u^2)_{xx}$$

A second-order-in-time dispersive wave equation with a nonlinear term containing second derivatives, $(u^2)_{xx} = 2u\,u_{xx} + 2u_x^2$, and a fourth-order linear term. For $|c|<1$ it has the exact soliton (a depression with this sign of the nonlinearity)

$$u(x,t) = -\tfrac32(1-c^2)\,\mathrm{sech}^2\!\big(\kappa(x - ct)\big),\qquad \kappa = \tfrac12\sqrt{1-c^2}$$

Initial velocity: $u_t(x,0) = -c\,u_x(x,0)$. The soliton is localized, so the periodic domain must be large (the tails decay like $e^{-2\kappa|x|}$).

In [ ]:
# Define the symbols and the Boussinesq equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

eq = Eq(diff(u, t, t) - diff(u, x, x) + diff(u, x, 4) - diff(u**2, x, 2), 0)   # u_tt = u_xx - u_xxxx + (u^2)_xx

# Soliton parameters
c = 0.6                                  # Speed (|c| < 1)
kappa = 0.5 * np.sqrt(1 - c**2)
A = -1.5 * (1 - c**2)                    # Amplitude (negative)

# Exact solution
def u_exact(x, t):
    return A / np.cosh(kappa * (x - c * t))**2

# Initial conditions: u(x,0) and u_t(x,0) = -c u_x(x,0)
initial_condition = lambda x: u_exact(x, 0.0)
initial_velocity = lambda x: 2 * kappa * c * A * np.tanh(kappa * x) / np.cosh(kappa * x)**2

# Create the solver (quadratic nonlinearity: 2/3 rule)
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 80
Nx = 512
Lt = 6.0
Nt = 4000

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
             initial_condition=initial_condition,
             initial_velocity=initial_velocity)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

## Benjamin–Ono equation

### Periodic soliton (nonlocal dispersion, `psiOp`)

$$u_t + u\,u_x + \mathcal H u_{xx} = 0\quad\Longleftrightarrow\quad u_t = -u\,u_x - \mathrm{Op}\big(i\,\xi\,|\xi|\big)\,u$$

$\mathcal H$ is the Hilbert transform (symbol $-i\,\mathrm{sgn}\,\xi$), so $\mathcal H\partial_x^2$ has symbol $i\xi|\xi|$: a **non-smooth symbol**, impossible to write with ordinary derivatives. The periodic soliton (Benjamin 1967) is a Poisson-kernel profile:

$$u(x,t) = a - 2\kappa\,\frac{\sinh\gamma}{\cosh\gamma - \cos\kappa(x - ct)},\qquad c = a - \kappa\coth\gamma$$

(the free constant $a$ is a Galilean shift; $a = 2\kappa$ gives a zero-mean wave). Derivation: with $P = \frac{\sinh\gamma}{\cosh\gamma - \cos\theta}$ and its conjugate $Q$, one has $P' = -PQ$, $\mathcal HP = Q$ and $Q^2 = -P^2 + 2\coth\gamma\,P - 1$, which closes the traveling-wave equation.

The `psiOp` integrator is exponential Euler: the nonlocal linear part is exact, the nonlinearity is first order in $\Delta t$ (error $\propto 1/N_t$: 9e-4, 4.6e-4, 2.3e-4 for $N_t$ = 6000, 12000, 24000 with $\gamma = 1.5$). A taller soliton (small $\gamma$) needs much smaller time steps: exponential Euler is unstable for an undamped dispersive nonlinearity when $\Delta t\,|u|\,k_{\max}$ is large.

In [ ]:
# Define the symbols and the Benjamin-Ono equation (psiOp form)
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

linear_symbol = I * xi * Abs(xi)                       # symbol of H d^2/dx^2
eq = Eq(diff(u(t, x), t), -u(t, x) * diff(u(t, x), x) - psiOp(linear_symbol, u(t, x)))   # u_t + u u_x + H u_xx = 0

# Soliton parameters
kappa = 1.0                  # Wavenumber (period 2 pi / kappa)
gamma = 1.5                  # Sharpness (small gamma = tall, narrow soliton)
a = 2 * kappa                # Galilean shift: zero-mean wave
c = a - kappa / np.tanh(gamma)   # Speed

# Exact solution
def u_exact(x, t):
    return a - 2 * kappa * np.sinh(gamma) / (np.cosh(gamma) - np.cos(kappa * (x - c * t)))

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (psiOp equations use the first-order exponential integrator)
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi / kappa       # Exactly one period
Nx = 64
Lt = 3.0
Nt = 24000                   # First order in time: the error halves each time Nt doubles

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-3, component='real') for i in range(n_test + 1)]

## Fokker–Planck equation 
### the Ornstein–Uhlenbeck process

$$u_t = D\,u_{xx} + (x\,u)_x = D\,u_{xx} + x\,u_x + u$$

A diffusion with a restoring drift: the coefficient of $u_x$ depends on $x$. A Gaussian stays Gaussian, with a relaxing mean and variance:

$$u(x,t) = \frac{1}{\sqrt{2\pi s^2(t)}}\exp\!\Big(-\frac{(x-m(t))^2}{2s^2(t)}\Big),\qquad m(t) = m_0 e^{-t},\quad s^2(t) = D + (s_0^2 - D)\,e^{-2t}$$

It relaxes towards the stationary law $\mathcal N(0, D)$. The eigenfunctions of this operator are Hermite functions (see the next notebook).
**Requires the variable-coefficient fix in `_parse_equation`** (zero-order terms such as $x^2u$ *and* derivative terms such as $x\,u_x$, $a(x)u_{xx}$). Without it, a zero-order variable coefficient is silently replaced by $1$ and a variable coefficient in front of a derivative crashes with a `TypeError`. With the fix these terms are evaluated pointwise, like nonlinear terms (explicit part of the integrator).

In [ ]:
# Define the symbols and the Fokker-Planck equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

D = 0.5
eq = Eq(diff(u, t) - D * diff(u, x, x) - diff(x * u, x), 0)   # u_t = D u_xx + (x u)_x

# Initial law parameters
m0, s0 = 1.5, 0.8   # Initial mean and standard deviation

# Exact solution
def u_exact(x, t):
    var = D + (s0**2 - D) * np.exp(-2 * t)
    m = m0 * np.exp(-t)
    return np.exp(-(x - m)**2 / (2 * var)) / np.sqrt(2 * np.pi * var)

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 30
Nx = 256
Lt = 3.0
Nt = 600

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-8, component='real') for i in range(n_test + 1)]

## Method of manufactured solutions

Pick any smooth $u^\star(x,t)$, let sympy compute the source $f$ it requires, and add $f$ to the equation: $u^\star$ is then the exact solution, whatever the nonlinearity. This is the most general way to exercise the nonlinear terms.

1. **1D**: a coefficient on $u_t$, explicit time dependence ($\sin t\,u^2$) and a convective term.
2. **2D**: a mixed derivative, $u\,u_{xy}$.

### 1D: $2\,u_t = 2\,(\nu u_{xx} - u u_x + \sin t\; u^2 + f)$

In [ ]:
# Define the symbols and the equation
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

nu = 0.05

# Manufactured solution and the source it requires
u_star = exp(-t) * (sin(x) + sin(2 * x + 1) / 2)
f = diff(u_star, t) - nu * diff(u_star, x, x) + u_star * diff(u_star, x) - sin(t) * u_star**2

eq = Eq(2 * diff(u, t) - 2 * (nu * diff(u, x, x) - u * diff(u, x) + sin(t) * u**2 + f), 0)

# Exact solution
# Exact solution
def u_exact(x, t):
    return np.exp(-t) * (np.sin(x) + np.sin(2 * x + 1) / 2)

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi
Nx = 64
Lt = 2.0
Nt = 200

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-8, component='real') for i in range(n_test + 1)]

### Variable coefficients: manufactured solutions

Three classical forms of variable-coefficient operators, with the exact solution $u^\star = e^{-t}(\sin x + \tfrac12\cos 2x)$ imposed through the source $f$ computed by sympy:

1. **Transport with variable speed**, $u_t + c(x)\,u_x = \nu\,u_{xx} + f$, $c = 1 + \tfrac12\sin x$.
2. **Variable diffusion**, $u_t = a(x)\,u_{xx} + f$, $a = 1 + \tfrac12\cos x$.
3. **Conservative form**, $u_t = (a(x)\,u_x)_x + f$.

The solver requires a non-null linear operator: the small Laplacian $\nu u_{xx}$ ($\nu = 0.05$) is mandatory in the transport case, and in the diffusion cases the constant part of $a(x)$ provides it (the solver splits $a(x)u_{xx} = u_{xx} + \tfrac12\cos x\,u_{xx}$: constant part exact, variable part explicit).
**Requires the variable-coefficient fix in `_parse_equation`** (zero-order terms such as $x^2u$ *and* derivative terms such as $x\,u_x$, $a(x)u_{xx}$). Without it, a zero-order variable coefficient is silently replaced by $1$ and a variable coefficient in front of a derivative crashes with a `TypeError`. With the fix these terms are evaluated pointwise, like nonlinear terms (explicit part of the integrator).

#### 1. **Transport with variable speed**, $u_t + c(x)\,u_x = \nu\,u_{xx} + f$, $c = 1 + \tfrac12\sin x$.

In [ ]:
# Define the symbols
t, x, kx, xi, omega = symbols('t x kx xi omega')
u_func = Function('u')
u = u_func(t, x)

# Manufactured solution (SymPy uniquement pour construire f et eq)
u_star = exp(-t) * (sin(x) + cos(2 * x) / 2)

# Exact solution & initial condition in pure NumPy (sans lambdify)
def u_exact(x, t):
    return np.exp(-t) * (np.sin(x) + np.cos(2 * x) / 2)

initial_condition = lambda x: np.sin(x) + np.cos(2 * x) / 2

# Variable speed, regularized by a small Laplacian
nu = 0.05
c = 1 + sin(x) / 2
f = diff(u_star, t) + c * diff(u_star, x) - nu * diff(u_star, x, x)
eq = Eq(diff(u, t) + c * diff(u, x) - nu * diff(u, x, x) - f, 0)   # u_t + c(x) u_x = nu u_xx + f

# Create the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# Simulation parameters
Lx = 2 * np.pi
Nx = 64
Lt = 1.0
Nt = 400

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing
n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

#### 2. **Variable diffusion**, $u_t = a(x)\,u_{xx} + f$, $a = 1 + \tfrac12\cos x$.

In [ ]:
# Manufactured solution (SymPy uniquement pour f et eq)
u_star = exp(-t) * (sin(x) + cos(2 * x) / 2)

# Exact solution & initial condition en NumPy pur
def u_exact(x, t):
    return np.exp(-t) * (np.sin(x) + np.cos(2 * x) / 2)

initial_condition = lambda x: np.sin(x) + np.cos(2 * x) / 2

# Variable diffusion coefficient
a = 1 + cos(x) / 2
f = diff(u_star, t) - a * diff(u_star, x, x)
eq = Eq(diff(u, t) - a * diff(u, x, x) - f, 0)   # u_t = a(x) u_xx + f

solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)
solver.solve()

n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]

#### 3. **Conservative form**, $u_t = (a(x)\,u_x)_x + f$.

In [ ]:
# Manufactured solution (SymPy uniquement pour f et eq)
u_star = exp(-t) * (sin(x) + cos(2 * x) / 2)

# Exact solution & initial condition en NumPy pur
def u_exact(x, t):
    return np.exp(-t) * (np.sin(x) + np.cos(2 * x) / 2)

initial_condition = lambda x: np.sin(x) + np.cos(2 * x) / 2

# Conservative (divergence) form
f = diff(u_star, t) - diff(a * diff(u_star, x), x)
eq = Eq(diff(u, t) - diff(a * diff(u, x), x) - f, 0)   # u_t = (a(x) u_x)_x + f

solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, initial_condition=initial_condition)
solver.solve()

n_test = 6
test_set = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]